# Eficiência das cotações de apostas esportivas

**Disciplina de Mineração de Dados — Tecnologia em Análise e Desenvolvimento de Sistemas**
**IFSP — Câmpus Jacareí**

Este notebook produz os números do resumo submetido à **JECET 2026** e implementa
cada etapa metodológica descrita nele. A organização segue as fases do
**CRISP-DM**: entendimento do negócio, entendimento dos dados, preparação,
modelagem, avaliação e implantação (aqui, a geração do relatório final).

## O que o trabalho quer responder

1. A **margem embutida nas cotações** (o *overround*) varia entre operadores e
   entre divisões?
2. As probabilidades atribuídas aos **desfechos menos prováveis** são
   superestimadas (viés favorito-azarão)?
3. Modelos supervisionados treinados apenas com **desempenho histórico**
   conseguem prever o resultado das partidas melhor do que o mercado?

## Regra de ouro deste notebook

Nenhum número é digitado à mão: tudo o que aparece no texto final vem da
execução. Se alguma afirmação do resumo **não se confirmar**, o notebook diz
isso com clareza — a análise nunca é ajustada para concordar com o texto.

---
## 1. Entendimento do negócio

Uma casa de apostas não publica probabilidades: publica **cotações** (*odds*).
Uma cotação de 2,50 significa que 1 unidade apostada devolve 2,50 se o palpite
acertar. O inverso da cotação, `1 / 2,50 = 0,40`, é a **probabilidade
implícita bruta**.

Se a casa fosse neutra, as três implícitas de uma partida (vitória do mandante,
empate, vitória do visitante) somariam exatamente 1. Na prática somam mais que
isso: a diferença é a **margem** (*overround*), a comissão embutida.

O trabalho trata as cotações como uma **previsão do mercado** e pergunta se
modelos de mineração de dados, treinados só com histórico de desempenho,
chegam perto dessa previsão.

In [ ]:
# Configuração do ambiente e das dependências do notebook.
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# O módulo src/utils.py concentra o código mecânico (download, Elo, médias
# móveis, métricas) para manter as células curtas e legíveis.
RAIZ = Path.cwd()
if not (RAIZ / "src").exists() and (RAIZ.parent / "src").exists():
    RAIZ = RAIZ.parent            # permite rodar com o notebook aberto em subpasta
sys.path.insert(0, str(RAIZ / "src"))
import utils

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
utils.estilo_padrao()

INICIO_EXECUCAO = time.time()
np.random.seed(utils.SEMENTE)

print("Python     :", sys.version.split()[0])
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("Semente    :", utils.SEMENTE)
print("Raiz       :", utils.RAIZ)

---
## 2. Entendimento dos dados

### 2.1 Fonte

A fonte é o **Football-Data.co.uk**, que publica, por temporada e divisão, um
CSV com resultado, estatísticas de jogo e cotações de vários operadores. O
padrão de URL é

```
https://www.football-data.co.uk/mmz4281/{temporada}/{divisao}.csv
```

com a temporada em quatro dígitos (`0506` para 2005/06, `2425` para 2024/25) e
o dicionário de colunas em <https://www.football-data.co.uk/notes.txt>.

**Atenção aos nomes de coluna**, que mudam ao longo das temporadas:

| Informação | Temporadas antigas | Temporadas recentes |
|---|---|---|
| Média do mercado | `BbAvH`, `BbAvD`, `BbAvA` | `AvgH`, `AvgD`, `AvgA` |
| Máxima do mercado | `BbMxH`, ... | `MaxH`, ... |
| Pinnacle | `PSH`, `PSD`, `PSA` | `PSH` (ou `PH`) |
| Cotação de fechamento | não existe | sufixo `C`: `B365CH`, ... |

O notebook **confirma** a presença de cada trio de colunas antes de usá-lo, em
vez de supor.

### 2.2 Ligas e temporadas

* **Ligas principais** (modelagem e todas as análises): `E0` Inglaterra,
  `SP1` Espanha, `I1` Itália, `D1` Alemanha, `F1` França.
* **Segundas divisões** (apenas na análise de margem por divisão): `E1`, `SP2`,
  `I2`, `D2`, `F2`.
* **2000/01 a 2004/05**: só aquecimento do Elo e das médias móveis, fora da
  amostra analisada.
* **2005/06 a 2024/25**: amostra analisada.

In [ ]:
print("Ligas principais :", utils.LIGAS_PRINCIPAIS)
print("Segundas divisões:", utils.LIGAS_SEGUNDAS)
print("Aquecimento      :", utils.rotulo_temporada(utils.ANOS_AQUECIMENTO[0]),
      "a", utils.rotulo_temporada(utils.ANOS_AQUECIMENTO[-1]),
      f"({len(utils.ANOS_AQUECIMENTO)} temporadas)")
print("Amostra analisada:", utils.rotulo_temporada(utils.ANOS_AMOSTRA[0]),
      "a", utils.rotulo_temporada(utils.ANOS_AMOSTRA[-1]),
      f"({len(utils.ANOS_AMOSTRA)} temporadas)")
print("Treino           :", utils.rotulo_temporada(utils.ANOS_TREINO[0]),
      "a", utils.rotulo_temporada(utils.ANOS_TREINO[-1]))
print("Teste            :", utils.rotulo_temporada(utils.ANOS_TESTE[0]),
      "a", utils.rotulo_temporada(utils.ANOS_TESTE[-1]))

### 2.3 Coleta

A pasta `data/raw/` funciona como **cache**: arquivo já baixado não é buscado
de novo. A função tenta primeiro a fonte oficial e, se o domínio estiver
inacessível a partir da máquina, usa um **espelho público** dos mesmos arquivos,
registrando a origem de cada um. Tudo o que foi baixado com sucesso e o que
falhou fica no relatório abaixo.

In [ ]:
# Baixa (ou reaproveita do cache) as 10 divisões x 25 temporadas.
relatorio_coleta = utils.baixar_temporadas(
    utils.LIGAS_PRINCIPAIS + utils.LIGAS_SEGUNDAS, utils.TEMPORADAS_TODAS)

print("Arquivos esperados:", len(utils.LIGAS_PRINCIPAIS + utils.LIGAS_SEGUNDAS)
      * len(utils.TEMPORADAS_TODAS))
print()
print("Situação de cada arquivo:")
print(relatorio_coleta["situacao"].value_counts().to_string())

falhas = relatorio_coleta[relatorio_coleta["situacao"] == "falhou"]
print(f"\nFalhas: {len(falhas)}")
if len(falhas):
    print(falhas[["divisao", "temporada", "detalhe"]].to_string(index=False))

if not relatorio_coleta.attrs["oficial_ok"]:
    print("\n" + "=" * 78)
    print("AVISO: a fonte oficial (football-data.co.uk) não está acessível "
          "deste ambiente.")
    print("Motivo:", relatorio_coleta.attrs["motivo_oficial"][:110])
    print("Os arquivos vieram do espelho público. Ver 'Decisões e limitações'.")
    print("=" * 78)

utils.salvar_tabela(relatorio_coleta, "coleta")

### 2.4 Leitura robusta

Os CSVs do Football-Data trazem três problemas recorrentes: codificação que
varia entre temporadas (UTF-8 nas recentes, Windows-1252 nas antigas), vírgulas
sobrando no fim das linhas (que criam colunas fantasma) e linhas em branco.

A leitura tenta **UTF-8 e cai para latin-1**, descarta colunas e linhas
totalmente vazias e **conta** as linhas malformadas em vez de sumir com elas em
silêncio.

In [ ]:
# Ligas principais: todas as temporadas (inclui o aquecimento 2000/01-2004/05).
base_principais, diagnostico_leitura = utils.carregar_base_bruta(
    utils.LIGAS_PRINCIPAIS, utils.TEMPORADAS_TODAS)

# Segundas divisões: só a amostra analisada, usadas apenas na análise de margem.
base_segundas, diagnostico_segundas = utils.carregar_base_bruta(
    utils.LIGAS_SEGUNDAS, utils.TEMPORADAS_AMOSTRA)

diagnostico = pd.concat([diagnostico_leitura, diagnostico_segundas],
                        ignore_index=True)

print(f"Ligas principais : {len(base_principais):,} linhas, "
      f"{base_principais.shape[1]} colunas".replace(",", "."))
print(f"Segundas divisões: {len(base_segundas):,} linhas, "
      f"{base_segundas.shape[1]} colunas".replace(",", "."))
print()
print("Codificação usada por arquivo:")
print(diagnostico["codificacao"].value_counts().to_string())
print()
print(f"Linhas totalmente vazias descartadas : "
      f"{int(diagnostico['linhas_vazias'].sum())}")
print(f"Linhas malformadas descartadas       : "
      f"{int(diagnostico['linhas_malformadas'].sum())}")
print(f"Colunas vazias descartadas           : "
      f"{int(diagnostico['colunas_descartadas'].sum())}")
print(f"Arquivos lidos sem problema          : "
      f"{int((diagnostico['situacao'] == 'ok').sum())} de {len(diagnostico)}")

utils.salvar_tabela(diagnostico, "leitura_diagnostico")

In [ ]:
# Partidas por temporada e liga: confere se a base tem o tamanho esperado.
contagem = (base_principais.groupby(["temporada", "Div"]).size()
            .unstack().reindex(columns=utils.LIGAS_PRINCIPAIS))
print("Partidas por temporada e liga (ligas principais):")
print(contagem.to_string())

Os números batem com a realidade das competições, o que é um bom indício de que
a base está íntegra: Bundesliga com 306 partidas (18 clubes), Premier League,
LaLiga e Serie A com 380 (20 clubes), Serie A com 306 até 2004/05 (quando ainda
tinha 18 clubes), Ligue 1 com 279 em 2019/20 (temporada interrompida pela
pandemia) e 306 a partir de 2023/24 (redução para 18 clubes).

---
## 3. Preparação dos dados

### 3.1 Datas

A coluna `Date` aparece em dois formatos na mesma base: `dd/mm/yy` nas
temporadas antigas e `dd/mm/yyyy` nas recentes. A conversão tenta os dois, na
ordem, e o notebook confere que **nenhuma data ficou sem converter**. Depois a
base é ordenada por data e, quando existe, por horário — ordenação que é a
espinha dorsal de todo o resto (médias móveis, Elo e divisão temporal).

In [ ]:
base_principais, relatorio_datas = utils.preparar_datas(base_principais)
base_segundas, relatorio_datas_segundas = utils.preparar_datas(base_segundas)

for chave, valor in relatorio_datas.items():
    print(f"{chave:28s}: {valor:,}".replace(",", "."))

assert relatorio_datas["datas_nao_convertidas"] == 0, "há datas não convertidas"
assert relatorio_datas_segundas["datas_nao_convertidas"] == 0

print()
print("Período coberto (ligas principais):",
      base_principais["data"].min().date(), "a", base_principais["data"].max().date())
print("A coluna de horário só existe a partir de 2019/20; por isso o desempate "
      "por horário vale apenas nas temporadas recentes.")

### 3.2 Padronização dos nomes das equipes

O resumo declara a padronização dos nomes das equipes como etapa da
metodologia, então ela ganha seção própria. São três verificações:

1. **Espaços sobrando** no início, no fim ou repetidos no meio do nome.
2. **Colisões de normalização**: nomes diferentes que viram a mesma chave depois
   de remover acentos, pontuação e caixa (ex.: `Ath Bilbao` vs `Ath. Bilbao`).
   São renomeações certas e entram no dicionário automaticamente.
3. **Pares parecidos** (similaridade ≥ 0,80), listados para **inspeção
   manual** — aqui é preciso cuidado, porque clubes homônimos legítimos
   aparecem nessa lista e **não** devem ser unidos.

In [ ]:
diagnostico_equipes = utils.diagnosticar_equipes(base_principais, limiar=0.80)

espacos_sobrando = int(
    base_principais["HomeTeam"].astype(str)
    .ne(utils.limpar_espacos(base_principais["HomeTeam"])).sum()
    + base_principais["AwayTeam"].astype(str)
    .ne(utils.limpar_espacos(base_principais["AwayTeam"])).sum())
print(f"Nomes com espaços sobrando: {espacos_sobrando}")
print()

for liga, info in diagnostico_equipes.items():
    print(f"{liga} ({utils.NOME_LIGA[liga]}): {info['n_nomes']} nomes distintos")
    print(f"   colisões de normalização : {info['colisoes'] if info['colisoes'] else 'nenhuma'}")
    print(f"   pares parecidos (≥ 0,80) : {info['candidatos'] if info['candidatos'] else 'nenhum'}")

In [ ]:
# Dicionário de mapeamento EXPLÍCITO, exibido no notebook.
#
# Parte automática: todas as colisões de normalização encontradas acima.
mapeamento_equipes = utils.mapeamento_por_colisao(diagnostico_equipes, base_principais)

# Parte manual: pares que a inspeção acima mostrou serem a mesma equipe.
# Fica vazio quando a inspeção não encontra nenhum — e é o que acontece aqui.
mapeamento_manual = {}
mapeamento_equipes.update(mapeamento_manual)

print("Dicionário de mapeamento de equipes:")
if mapeamento_equipes:
    for origem, destino in sorted(mapeamento_equipes.items()):
        print(f"   {origem!r} -> {destino!r}")
else:
    print("   (vazio)")

base_principais = utils.aplicar_mapeamento_equipes(base_principais, mapeamento_equipes)
base_segundas = utils.aplicar_mapeamento_equipes(base_segundas, mapeamento_equipes)

pd.DataFrame(sorted(mapeamento_equipes.items()) or [("(vazio)", "(vazio)")],
             columns=["grafia_original", "grafia_padronizada"]).pipe(
    utils.salvar_tabela, "mapeamento_equipes")

**Resultado da padronização.** A limpeza de espaços é aplicada sempre. Já o
dicionário de renomeações saiu **vazio**: o Football-Data.co.uk usa nomes curtos
já consistentes entre temporadas (`Man United`, `Ath Bilbao`, `M'gladbach`), e
não há colisão de normalização em nenhuma das cinco ligas principais.

Os únicos pares parecidos que a busca aponta são clubes **diferentes** —
`Piacenza` e `Vicenza` na Itália, `Le Mans` e `Lens` na França, e na França
ainda `Ajaccio` (AC Ajaccio) e `Ajaccio GFCO` (GFC Ajaccio), dois clubes da
mesma cidade. Uni-los seria um erro, então o mapeamento manual fica vazio.

Registramos isso porque é um resultado honesto da etapa, e não uma etapa pulada.

### 3.3 Cobertura das colunas relevantes

Antes de decidir o que usar, é preciso saber o que existe. A tabela de cobertura
mostra o **percentual de valores não nulos** por liga e temporada para gols,
finalizações, escanteios, cartões e as cotações de cada operador. Valor ausente
(`NaN` na tabela) significa que a **coluna não existe** naquela temporada, que é
diferente de existir e estar vazia.

In [ ]:
# Colunas de cotação de todos os operadores investigados.
colunas_cotacoes = []
for prefixos in utils.OPERADORES.values():
    for prefixo in prefixos:
        colunas_cotacoes += [f"{prefixo}{s}" for s in ("H", "D", "A")]
colunas_cotacoes = [c for c in dict.fromkeys(colunas_cotacoes)
                    if c in base_principais.columns or c in base_segundas.columns]

colunas_relevantes = (utils.COLUNAS_RESULTADO + utils.COLUNAS_ESTATISTICAS
                      + colunas_cotacoes)

cobertura = utils.tabela_cobertura(
    pd.concat([base_principais, base_segundas], ignore_index=True), colunas_relevantes)
caminho = utils.salvar_tabela(cobertura, "cobertura")
print("Tabela de cobertura salva em:", caminho.relative_to(utils.RAIZ))
print(f"Linhas: {len(cobertura)} (liga x temporada), colunas: {cobertura.shape[1]}")
print()

# Resumo por temporada nas ligas principais: gols, estatísticas e operadores.
resumo_cobertura = (cobertura[cobertura["liga"].isin(utils.LIGAS_PRINCIPAIS)]
                    .groupby("temporada")[["FTHG", "HS", "HST", "HC",
                                           "B365H", "BWH", "IWH", "PSH",
                                           "WHH", "VCH", "BbAvH", "AvgH"]]
                    .mean().round(1))
print("Cobertura média (%) por temporada, ligas principais:")
print(resumo_cobertura.to_string())

A tabela deixa ver três coisas que orientam as decisões seguintes:

* **Bet365** (`B365H`) cobre praticamente 100% das partidas de 2005/06 em
  diante — por isso é o operador principal do trabalho.
* A **média de mercado** muda de nome: `BbAvH` até 2018/19, `AvgH` de 2019/20 em
  diante. O notebook trata as duas como o mesmo operador.
* As **estatísticas de jogo** (finalizações, escanteios) só ficam completas a
  partir de 2005/06 — nas temporadas de aquecimento a cobertura é parcial, o que
  afeta apenas o início das médias móveis.

### 3.4 Filtros e N final

Saem da base as partidas **sem resultado** e as **sem cotações da Bet365**,
com a contagem registrada. O N que sobra nas ligas principais em 2005/06–2024/25
é o número que substitui o "cerca de quarenta mil" do resumo.

In [ ]:
COLUNAS_B365 = utils.colunas_operador(base_principais, utils.OPERADORES["Bet365"])
print("Colunas da Bet365 confirmadas na base:", COLUNAS_B365)

def aplicar_filtros(base, rotulo):
    relatorio = {"rotulo": rotulo, "linhas_iniciais": len(base)}

    sem_resultado = ~base["FTR"].isin(["H", "D", "A"])
    relatorio["sem_resultado"] = int(sem_resultado.sum())
    base = base[~sem_resultado].copy()

    probabilidades = utils.probabilidades_normalizadas(base, COLUNAS_B365)
    sem_cotacao = probabilidades.isna().any(axis=1)
    relatorio["sem_cotacao_bet365"] = int(sem_cotacao.sum())
    base = base[~sem_cotacao.to_numpy()].copy()

    relatorio["linhas_finais"] = len(base)
    return base, relatorio

# As temporadas de aquecimento entram sem o filtro de cotação: elas só servem
# para construir histórico de desempenho, e a Bet365 só aparece em 2002/03.
aquecimento = base_principais[base_principais["ano_temporada"] < 2005]
amostra_bruta = base_principais[base_principais["ano_temporada"] >= 2005]

aquecimento_limpo = aquecimento[aquecimento["FTR"].isin(["H", "D", "A"])].copy()
amostra_limpa, rel_amostra = aplicar_filtros(amostra_bruta, "amostra 2005/06-2024/25")
base_segundas, rel_segundas = aplicar_filtros(base_segundas, "segundas divisões")

for chave, valor in rel_amostra.items():
    print(f"{chave:24s}: {valor}")
print()
print(f"Aquecimento (2000/01-2004/05): {len(aquecimento_limpo):,} partidas, "
      f"{int((~aquecimento['FTR'].isin(['H','D','A'])).sum())} sem resultado"
      .replace(",", "."))
print()
print("=" * 70)
N_AMOSTRA = len(amostra_limpa)
print(f"N FINAL de partidas das ligas principais em 2005/06-2024/25: "
      f"{N_AMOSTRA:,}".replace(",", "."))
print("=" * 70)
print("Este número substitui o 'cerca de quarenta mil' do resumo.")

# Base completa usada para construir histórico: aquecimento + amostra.
base_historico = pd.concat([aquecimento_limpo, amostra_limpa], ignore_index=True)
base_historico = base_historico.sort_values(
    ["data", "horario", "Div", "HomeTeam"], kind="mergesort").reset_index(drop=True)
print(f"\nBase para construção de histórico (com aquecimento): "
      f"{len(base_historico):,} partidas".replace(",", "."))

### 3.5 Das cotações para probabilidades

Para cada partida, com as cotações da Bet365:

* **Probabilidade implícita bruta**: `1 / cotação`.
* **Margem** (*overround*): soma das três implícitas menos 1.
* **Probabilidade normalizada**: cada implícita dividida pela soma das três —
  é a implícita com a margem removida, e é ela que serve de previsão do mercado.
* **Favorito do mercado**: o desfecho com maior probabilidade normalizada.

In [ ]:
implicitas_b365, margem_b365 = utils.implicitas_e_margem(amostra_limpa, COLUNAS_B365)
probabilidades_b365 = utils.probabilidades_normalizadas(amostra_limpa, COLUNAS_B365)

mercado = amostra_limpa[["partida_id", "Div", "ano_temporada", "data", "FTR"]].copy()
mercado = mercado.reset_index(drop=True)
mercado[["prob_H", "prob_D", "prob_A"]] = probabilidades_b365.to_numpy()
mercado["margem"] = margem_b365.to_numpy()
mercado[["cot_H", "cot_D", "cot_A"]] = amostra_limpa[COLUNAS_B365].to_numpy()
mercado["favorito"] = np.array(utils.CLASSES)[
    mercado[["prob_H", "prob_D", "prob_A"]].to_numpy().argmax(axis=1)]

print("Exemplo — primeira partida da amostra:")
exemplo = mercado.iloc[0]
print(f"   cotações Bet365 (H/D/A) : {exemplo['cot_H']}, {exemplo['cot_D']}, {exemplo['cot_A']}")
print(f"   implícitas brutas       : "
      f"{1/exemplo['cot_H']:.4f}, {1/exemplo['cot_D']:.4f}, {1/exemplo['cot_A']:.4f}")
print(f"   soma das implícitas     : {1 + exemplo['margem']:.4f}")
print(f"   margem (overround)      : {exemplo['margem']:.4f} "
      f"({100*exemplo['margem']:.2f}%)")
print(f"   normalizadas            : {exemplo['prob_H']:.4f}, "
      f"{exemplo['prob_D']:.4f}, {exemplo['prob_A']:.4f}")
print(f"   soma das normalizadas   : "
      f"{exemplo[['prob_H','prob_D','prob_A']].sum():.6f}")
print(f"   favorito do mercado     : {exemplo['favorito']}  (resultado: {exemplo['FTR']})")

assert np.allclose(mercado[["prob_H", "prob_D", "prob_A"]].sum(axis=1), 1.0)
print("\nConferido: as probabilidades normalizadas somam 1 em todas as partidas.")
print(f"Margem média da Bet365 na amostra: {100*mercado['margem'].mean():.2f}%")
print(f"Acerto do favorito do mercado    : "
      f"{100*(mercado['favorito'] == mercado['FTR']).mean():.2f}%")

---
## 4. Análise do mercado

Esta seção sustenta (ou refuta) a primeira frase de resultados do resumo:

> *"a margem embutida nas cotações varia de maneira consistente entre operadores
> e divisões e [...] as probabilidades atribuídas aos desfechos menos prováveis
> tendem a ser superestimadas"*

São duas afirmações independentes, testadas separadamente nas subseções 4.1–4.2
(margem) e 4.3–4.5 (superestimação dos desfechos improváveis).

### 4.1 Margem por operador

A margem de cada operador é calculada partida a partida e resumida pela média de
cada temporada. Um operador só entra numa temporada se tiver **cobertura de pelo
menos 80%** das partidas daquela temporada — assim a média não fica distorcida
por um punhado de jogos. Operadores entram e saem da base ao longo dos anos, e
as linhas do gráfico simplesmente começam e terminam quando há dados.

In [ ]:
COBERTURA_MINIMA = 0.80

def margem_do_operador(base, prefixos):
    # Alguns operadores mudam de prefixo entre temporadas (ex.: BbAv -> Avg).
    # Calculamos a margem para cada prefixo existente e combinamos.
    margem_total = None
    for prefixo in prefixos:
        colunas = [f"{prefixo}{s}" for s in ("H", "D", "A")]
        if not all(c in base.columns for c in colunas):
            continue
        _, margem = utils.implicitas_e_margem(base, colunas)
        margem_total = margem if margem_total is None else margem_total.combine_first(margem)
    return margem_total

linhas_margem = []
for operador, prefixos in utils.OPERADORES.items():
    margem = margem_do_operador(amostra_limpa, prefixos)
    if margem is None:
        print(f"{operador}: nenhuma coluna encontrada na base")
        continue
    tabela = pd.DataFrame({"ano_temporada": amostra_limpa["ano_temporada"].to_numpy(),
                           "Div": amostra_limpa["Div"].to_numpy(),
                           "margem": margem.to_numpy()})
    for ano, grupo in tabela.groupby("ano_temporada"):
        cobertura = grupo["margem"].notna().mean()
        if cobertura < COBERTURA_MINIMA:
            continue
        linhas_margem.append({
            "operador": operador, "ano_temporada": ano,
            "temporada": utils.rotulo_temporada(ano),
            "n_partidas": int(grupo["margem"].notna().sum()),
            "cobertura_pct": round(100 * cobertura, 1),
            "margem_media_pct": round(100 * grupo["margem"].mean(), 3),
            "margem_mediana_pct": round(100 * grupo["margem"].median(), 3)})

margem_operador = pd.DataFrame(linhas_margem)
utils.salvar_tabela(margem_operador, "margem_por_operador")

resumo_operador = (margem_operador.groupby("operador")
                   .agg(temporadas=("ano_temporada", "nunique"),
                        primeira=("ano_temporada", "min"),
                        ultima=("ano_temporada", "max"),
                        margem_media_pct=("margem_media_pct", "mean"))
                   .round(2).sort_values("margem_media_pct"))
print("Margem média por operador (temporadas com cobertura >= 80%):")
print(resumo_operador.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))

# A cor acompanha o operador (ordem fixa dos slots), nunca a sua posição
# no ranking: quem lê aprende "Bet365 é azul" e isso não muda entre gráficos.
ordem_operadores = list(utils.OPERADORES.keys())
cores = {op: utils.CORES_SERIES[i] for i, op in enumerate(ordem_operadores)}

for operador in ordem_operadores:
    serie = margem_operador[margem_operador["operador"] == operador].sort_values("ano_temporada")
    if serie.empty:
        continue
    ax.plot(serie["ano_temporada"], serie["margem_media_pct"],
            marker="o", markersize=3.5, linewidth=1.8,
            color=cores[operador], label=operador)

ax.set_title("Margem média embutida nas cotações, por operador e temporada")
ax.set_xlabel("Temporada (ano de início)")
ax.set_ylabel("Margem média (%)")
ax.set_xticks(range(2005, 2025, 2))
ax.set_xticklabels([utils.rotulo_temporada(a) for a in range(2005, 2025, 2)],
                   rotation=45, ha="right")
ax.legend(ncol=2, loc="upper right")
fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "margem_por_operador").relative_to(utils.RAIZ))
plt.show()

In [ ]:
# A margem varia entre operadores? Comparação na janela em que todos coexistem.
ultimo_ano_comum = (margem_operador.groupby("operador")["ano_temporada"].max().min())
primeiro_ano_comum = (margem_operador.groupby("operador")["ano_temporada"].min().max())
janela = margem_operador[
    margem_operador["ano_temporada"].between(primeiro_ano_comum, ultimo_ano_comum)]

comparacao = (janela.groupby("operador")["margem_media_pct"]
              .agg(["mean", "min", "max"]).round(2)
              .sort_values("mean"))
comparacao.columns = ["média", "mínimo", "máximo"]
print(f"Janela em que todos os operadores coexistem: "
      f"{utils.rotulo_temporada(primeiro_ano_comum)} a "
      f"{utils.rotulo_temporada(ultimo_ano_comum)}")
print(comparacao.to_string())

amplitude_operadores = comparacao["média"].max() - comparacao["média"].min()
razao_operadores = comparacao["média"].max() / comparacao["média"].min()
print(f"\nMenor margem média : {comparacao['média'].idxmin()} "
      f"({comparacao['média'].min():.2f}%)")
print(f"Maior margem média : {comparacao['média'].idxmax()} "
      f"({comparacao['média'].max():.2f}%)")
print(f"Amplitude entre operadores: {amplitude_operadores:.2f} pontos percentuais "
      f"({razao_operadores:.2f}x)")

### 4.2 Margem por divisão

Mesma conta, agora comparando a **primeira e a segunda divisão de cada país**,
com a Bet365 (o único operador que cobre todo o período nas dez divisões).
A pergunta é se apostar na divisão de acesso sai mais caro para o apostador.

In [ ]:
divisoes = pd.concat([
    amostra_limpa.assign(margem=utils.implicitas_e_margem(amostra_limpa, COLUNAS_B365)[1]),
    base_segundas.assign(margem=utils.implicitas_e_margem(base_segundas, COLUNAS_B365)[1]),
], ignore_index=True)

margem_divisao = (divisoes.groupby("Div")
                  .agg(n_partidas=("margem", "count"),
                       margem_media_pct=("margem", lambda s: 100 * s.mean()),
                       margem_mediana_pct=("margem", lambda s: 100 * s.median()))
                  .reset_index())
margem_divisao["pais"] = margem_divisao["Div"].map(utils.PAIS_LIGA)
margem_divisao["divisao"] = margem_divisao["Div"].map(utils.DIVISAO_LIGA)
margem_divisao["liga"] = margem_divisao["Div"].map(utils.NOME_LIGA)
margem_divisao = margem_divisao.round(3)
utils.salvar_tabela(margem_divisao, "margem_por_divisao")

tabela_paises = margem_divisao.pivot(index="pais", columns="divisao",
                                     values="margem_media_pct")
tabela_paises["diferença (2ª - 1ª)"] = (tabela_paises["2ª divisão"]
                                        - tabela_paises["1ª divisão"])
print("Margem média da Bet365 (%), 2005/06 a 2024/25:")
print(tabela_paises.round(2).to_string())
print(f"\nDiferença média (2ª - 1ª divisão): "
      f"{tabela_paises['diferença (2ª - 1ª)'].mean():.2f} pontos percentuais")
print(f"Países em que a 2ª divisão tem margem maior: "
      f"{int((tabela_paises['diferença (2ª - 1ª)'] > 0).sum())} de {len(tabela_paises)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

paises = list(tabela_paises.index)
posicao = np.arange(len(paises))
largura = 0.38
# Duas categorias -> dois primeiros slots da paleta. O vão de 2 px entre as
# barras vizinhas evita que as duas cores se toquem.
for deslocamento, divisao, cor in ((-largura/2 - 0.01, "1ª divisão", utils.CORES_SERIES[0]),
                                   (+largura/2 + 0.01, "2ª divisão", utils.CORES_SERIES[1])):
    valores = tabela_paises[divisao].to_numpy()
    barras = ax.bar(posicao + deslocamento, valores, largura,
                    label=divisao, color=cor)
    # Rótulo direto em cada barra: são poucas, e a leitura fica exata.
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.05,
                utils.formatar_br(valor, 2), ha="center", va="bottom",
                fontsize=8, color=utils.TINTA_SECUNDARIA)

ax.set_title("Margem média da Bet365 por país e divisão (2005/06–2024/25)")
ax.set_xlabel("País")
ax.set_ylabel("Margem média (%)")
ax.set_xticks(posicao)
ax.set_xticklabels(paises)
ax.legend()
ax.set_ylim(0, tabela_paises[["1ª divisão", "2ª divisão"]].to_numpy().max() * 1.18)
fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "margem_por_divisao").relative_to(utils.RAIZ))
plt.show()

### 4.3 Calibração das probabilidades

Aqui muda a unidade de análise. Cada partida vira **três observações**
empilhadas — uma por desfecho — com a probabilidade normalizada da Bet365 e a
informação de o desfecho ter ocorrido ou não.

As observações são agrupadas em faixas de 10 pontos percentuais. Em um mercado
bem calibrado, entre os desfechos cotados a ~20%, aproximadamente 20% deveriam
de fato acontecer. O intervalo de confiança de 95% é o de **Wilson**, mais
adequado que o normal para proporções perto de 0 e de 1.

In [ ]:
def empilhar_desfechos(tabela):
    # Três linhas por partida: (probabilidade prevista, ocorreu ou não, cotação).
    partes = []
    for classe in utils.CLASSES:
        partes.append(pd.DataFrame({
            "partida_id": tabela["partida_id"].to_numpy(),
            "ano_temporada": tabela["ano_temporada"].to_numpy(),
            "desfecho": classe,
            "prob": tabela[f"prob_{classe}"].to_numpy(),
            "cotacao": tabela[f"cot_{classe}"].to_numpy(),
            "ocorreu": (tabela["FTR"].to_numpy() == classe).astype(int)}))
    return pd.concat(partes, ignore_index=True)

empilhado = empilhar_desfechos(mercado)
print(f"Observações empilhadas: {len(empilhado):,}".replace(",", "."),
      f"= {len(mercado):,}".replace(",", "."), "partidas x 3 desfechos")

faixas = np.arange(0, 1.01, 0.10)
empilhado["faixa"] = pd.cut(empilhado["prob"], bins=faixas, include_lowest=True)

calibracao = (empilhado.groupby("faixa", observed=True)
              .agg(n_observacoes=("ocorreu", "size"),
                   prob_media_prevista=("prob", "mean"),
                   frequencia_observada=("ocorreu", "mean"),
                   sucessos=("ocorreu", "sum"))
              .reset_index())
limite_inferior, limite_superior = utils.ic_wilson(
    calibracao["sucessos"], calibracao["n_observacoes"])
calibracao["ic95_inferior"] = limite_inferior
calibracao["ic95_superior"] = limite_superior
calibracao["diferenca_pp"] = 100 * (calibracao["frequencia_observada"]
                                    - calibracao["prob_media_prevista"])
calibracao["faixa"] = calibracao["faixa"].astype(str)
utils.salvar_tabela(calibracao.round(5), "calibracao")

print()
print(calibracao.assign(
    prob_media_prevista=lambda d: (100*d.prob_media_prevista).round(2),
    frequencia_observada=lambda d: (100*d.frequencia_observada).round(2),
    ic95_inferior=lambda d: (100*d.ic95_inferior).round(2),
    ic95_superior=lambda d: (100*d.ic95_superior).round(2),
    diferenca_pp=lambda d: d.diferenca_pp.round(2),
)[["faixa", "n_observacoes", "prob_media_prevista", "frequencia_observada",
   "ic95_inferior", "ic95_superior", "diferenca_pp"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.4))

# Diagonal de referência: linha fina e cinza, claramente recessiva em relação
# aos dados.
ax.plot([0, 100], [0, 100], color=utils.TINTA_SUAVE, linewidth=1.0,
        zorder=1, label="Calibração perfeita")

x = 100 * calibracao["prob_media_prevista"].to_numpy()
y = 100 * calibracao["frequencia_observada"].to_numpy()
erro_baixo = y - 100 * calibracao["ic95_inferior"].to_numpy()
erro_alto = 100 * calibracao["ic95_superior"].to_numpy() - y

ax.errorbar(x, y, yerr=[erro_baixo, erro_alto], fmt="o-", markersize=5,
            linewidth=1.8, color=utils.CORES_SERIES[0], capsize=3, zorder=3,
            label="Bet365 (probabilidade normalizada)")

ax.set_title("Calibração das probabilidades da Bet365\n(faixas de 10 pontos percentuais, IC 95% de Wilson)")
ax.set_xlabel("Probabilidade média prevista (%)")
ax.set_ylabel("Frequência observada (%)")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_aspect("equal")
ax.legend(loc="upper left")
fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "calibracao").relative_to(utils.RAIZ))
plt.show()

### 4.4 Viés favorito-azarão

O teste da segunda afirmação do resumo. Duas leituras complementares:

1. **Pelas faixas.** Nas faixas de baixa probabilidade, a frequência observada
   fica **abaixo** da prevista? Se ficar, e o intervalo de confiança não
   cruzar a previsão, o mercado superestima os desfechos improváveis.
2. **Por regressão.** Regressão logística do desfecho sobre o **logit da
   probabilidade normalizada**. Inclinação **maior que 1** indica o viés: a
   probabilidade real responde de forma mais acentuada do que a cotada, ou seja,
   as probabilidades cotadas estão comprimidas em direção ao meio.

Como cada partida contribui com três observações correlacionadas, os erros
padrão são agrupados por partida (*cluster-robust*).

In [ ]:
# 1) Leitura pelas faixas de baixa probabilidade (abaixo de 30%).
faixas_baixas = calibracao[calibracao["prob_media_prevista"] < 0.30].copy()
print("Faixas de baixa probabilidade:")
for _, linha in faixas_baixas.iterrows():
    prevista = 100 * linha["prob_media_prevista"]
    observada = 100 * linha["frequencia_observada"]
    inferior, superior = 100 * linha["ic95_inferior"], 100 * linha["ic95_superior"]
    superestimado = superior < prevista      # IC inteiramente abaixo da previsão
    print(f"   {linha['faixa']:>14s} | n={int(linha['n_observacoes']):>6,} | "
          f"prevista {prevista:5.2f}% | observada {observada:5.2f}% "
          f"[{inferior:5.2f}; {superior:5.2f}] | "
          f"{'SUPERESTIMADA' if superestimado else 'dentro do IC'}"
          .replace(",", "."))

n_superestimadas = int((faixas_baixas["ic95_superior"]
                        < faixas_baixas["prob_media_prevista"]).sum())
print(f"\nFaixas de baixa probabilidade com superestimação significativa: "
      f"{n_superestimadas} de {len(faixas_baixas)}")

In [ ]:
# 2) Regressão logística do desfecho sobre o logit da probabilidade cotada.
import statsmodels.api as sm

ajuste = empilhado[(empilhado["prob"] > 1e-6) & (empilhado["prob"] < 1 - 1e-6)].copy()
ajuste["logito"] = np.log(ajuste["prob"] / (1 - ajuste["prob"]))

X_logit = sm.add_constant(ajuste[["logito"]])
modelo_logit = sm.Logit(ajuste["ocorreu"], X_logit).fit(
    disp=False, cov_type="cluster",
    cov_kwds={"groups": ajuste["partida_id"]})

inclinacao = modelo_logit.params["logito"]
ic_inclinacao = modelo_logit.conf_int().loc["logito"].to_numpy()
intercepto = modelo_logit.params["const"]

print(modelo_logit.summary2().tables[1].round(4).to_string())
print()
print(f"Inclinação estimada : {inclinacao:.4f}  "
      f"IC 95% [{ic_inclinacao[0]:.4f}; {ic_inclinacao[1]:.4f}]")
print(f"Intercepto          : {intercepto:.4f}")
print()
if ic_inclinacao[0] > 1:
    veredicto_regressao = "inclinação significativamente MAIOR que 1 -> viés presente"
elif ic_inclinacao[1] < 1:
    veredicto_regressao = "inclinação significativamente MENOR que 1 -> viés no sentido oposto"
else:
    veredicto_regressao = "IC contém 1 -> sem evidência de viés"
print("Veredicto:", veredicto_regressao)

VIES_CONFIRMADO_REGRESSAO = bool(ic_inclinacao[0] > 1)
VIES_CONFIRMADO_FAIXAS = bool(n_superestimadas > 0)

### 4.5 Retorno médio por faixa de cotação

A leitura mais didática do trabalho, e a mais próxima da experiência de quem
aposta. A simulação é simples: **1 unidade em cada desfecho de cada partida**,
com as cotações da Bet365. Quem acerta recebe `cotação − 1`; quem erra perde 1.

Num mercado sem margem e sem viés, o retorno médio seria 0 em todas as faixas.
Com margem, espera-se um retorno **negativo e parecido** em todas elas. Se o
retorno cair sistematicamente nas cotações altas, é o viés favorito-azarão
aparecendo em dinheiro.

O intervalo de confiança usa erro padrão agrupado por partida, pelo mesmo
motivo da seção anterior.

In [ ]:
limites_faixas = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0, np.inf]
rotulos_faixas = ["até 1,5", "1,5 a 2", "2 a 3", "3 a 5", "5 a 10", "acima de 10"]

apostas = empilhado.dropna(subset=["cotacao"]).copy()
apostas["retorno"] = np.where(apostas["ocorreu"] == 1, apostas["cotacao"] - 1.0, -1.0)
apostas["faixa_cotacao"] = pd.cut(apostas["cotacao"], bins=limites_faixas,
                                  labels=rotulos_faixas, right=False)

def erro_padrao_agrupado(grupo):
    # Erro padrão do retorno médio agrupando por partida (3 apostas por jogo).
    por_partida = grupo.groupby("partida_id")["retorno"].mean()
    n_grupos = len(por_partida)
    if n_grupos < 2:
        return np.nan
    return por_partida.std(ddof=1) / np.sqrt(n_grupos)

linhas_retorno = []
for faixa, grupo in apostas.groupby("faixa_cotacao", observed=True):
    media = grupo["retorno"].mean()
    erro = erro_padrao_agrupado(grupo)
    linhas_retorno.append({
        "faixa_cotacao": str(faixa),
        "n_apostas": len(grupo),
        "cotacao_media": grupo["cotacao"].mean(),
        "prob_media_prevista": grupo["prob"].mean(),
        "frequencia_acerto": grupo["ocorreu"].mean(),
        "retorno_medio": media,
        "ic95_inferior": media - 1.96 * erro,
        "ic95_superior": media + 1.96 * erro})

retorno = pd.DataFrame(linhas_retorno)
retorno["faixa_cotacao"] = pd.Categorical(retorno["faixa_cotacao"],
                                          categories=rotulos_faixas, ordered=True)
retorno = retorno.sort_values("faixa_cotacao").reset_index(drop=True)
utils.salvar_tabela(retorno.round(5), "retorno_por_faixa_cotacao")

print(retorno.assign(
    cotacao_media=lambda d: d.cotacao_media.round(2),
    prob_media_prevista=lambda d: (100*d.prob_media_prevista).round(2),
    frequencia_acerto=lambda d: (100*d.frequencia_acerto).round(2),
    retorno_medio=lambda d: (100*d.retorno_medio).round(2),
    ic95_inferior=lambda d: (100*d.ic95_inferior).round(2),
    ic95_superior=lambda d: (100*d.ic95_superior).round(2),
).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))

y = 100 * retorno["retorno_medio"].to_numpy()
erro = np.vstack([y - 100 * retorno["ic95_inferior"].to_numpy(),
                  100 * retorno["ic95_superior"].to_numpy() - y])
posicao = np.arange(len(retorno))

# Uma série -> uma cor (slot 1) para todas as barras. Colorir por valor
# duplicaria no matiz a informação que a altura da barra já dá.
ax.bar(posicao, y, 0.62, color=utils.CORES_SERIES[0], zorder=2)
ax.errorbar(posicao, y, yerr=erro, fmt="none", ecolor=utils.TINTA_PRIMARIA,
            elinewidth=1.2, capsize=4, zorder=3)
ax.axhline(0, color=utils.TINTA_SUAVE, linewidth=1.0, zorder=1)

for posicao_barra, valor in zip(posicao, y):
    deslocamento = -1.6 if valor < 0 else 1.0
    ax.text(posicao_barra, valor + deslocamento, utils.formatar_br(valor, 1) + "%",
            ha="center", va="top" if valor < 0 else "bottom",
            fontsize=9, color=utils.TINTA_SECUNDARIA)

ax.set_title("Retorno médio de apostar 1 unidade em cada desfecho (Bet365)\n"
             "por faixa de cotação, com IC 95%")
ax.set_xlabel("Faixa de cotação")
ax.set_ylabel("Retorno médio por unidade apostada (%)")
ax.set_xticks(posicao)
ax.set_xticklabels(retorno["faixa_cotacao"].astype(str))
fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "retorno_por_faixa_cotacao").relative_to(utils.RAIZ))
plt.show()

### 4.6 Cotação de pré-jogo contra cotação de fechamento

A cotação de **fechamento** é a última registrada antes da bola rolar, depois de
o mercado absorver notícias de escalação, lesões e o próprio fluxo de apostas.
A literatura a trata como a previsão mais informada disponível.

As colunas com sufixo `C` (`B365CH`, `B365CD`, `B365CA`) só existem em parte das
temporadas, então a comparação se restringe a elas. A métrica é o **log loss**
das probabilidades normalizadas: quanto menor, melhor.

In [ ]:
# O log loss é calculado por utils.log_loss_ordenado, e não pelo scikit-learn:
# o sklearn ordena o argumento "labels" alfabeticamente e pressupõe que as
# colunas de y_pred sigam a ordem A, D, H. Como este notebook usa a ordem fixa
# H, D, A, a função do sklearn trocaria as colunas de mandante e visitante.

COLUNAS_B365_FECHAMENTO = ["B365CH", "B365CD", "B365CA"]
tem_fechamento = all(c in amostra_limpa.columns for c in COLUNAS_B365_FECHAMENTO)
print("Colunas de fechamento presentes na base:", tem_fechamento)

if tem_fechamento:
    prob_fechamento = utils.probabilidades_normalizadas(
        amostra_limpa, COLUNAS_B365_FECHAMENTO)
    comparavel = prob_fechamento.notna().all(axis=1).to_numpy()
    temporadas_com = sorted(amostra_limpa.loc[comparavel, "ano_temporada"].unique())
    print(f"Partidas com cotação de fechamento: {int(comparavel.sum()):,}"
          .replace(",", "."))
    print("Temporadas cobertas:",
          ", ".join(utils.rotulo_temporada(a) for a in temporadas_com))

    y_comparavel = mercado.loc[comparavel, "FTR"].to_numpy()
    pre_jogo = mercado.loc[comparavel, ["prob_H", "prob_D", "prob_A"]].to_numpy()
    fechamento = prob_fechamento[comparavel].to_numpy()

    ll_pre = utils.log_loss_ordenado(y_comparavel, pre_jogo)
    ll_fec = utils.log_loss_ordenado(y_comparavel, fechamento)
    acc_pre = (np.array(utils.CLASSES)[pre_jogo.argmax(1)] == y_comparavel).mean()
    acc_fec = (np.array(utils.CLASSES)[fechamento.argmax(1)] == y_comparavel).mean()

    comparacao_fechamento = pd.DataFrame([
        {"conjunto": "pré-jogo", "n": int(comparavel.sum()),
         "log_loss": ll_pre, "acuracia": acc_pre,
         "margem_media_pct": 100 * utils.implicitas_e_margem(
             amostra_limpa[comparavel], COLUNAS_B365)[1].mean()},
        {"conjunto": "fechamento", "n": int(comparavel.sum()),
         "log_loss": ll_fec, "acuracia": acc_fec,
         "margem_media_pct": 100 * utils.implicitas_e_margem(
             amostra_limpa[comparavel], COLUNAS_B365_FECHAMENTO)[1].mean()},
    ])
    utils.salvar_tabela(comparacao_fechamento.round(6), "pre_jogo_vs_fechamento")
    print()
    print(comparacao_fechamento.round(4).to_string(index=False))
    print()
    if ll_fec < ll_pre:
        print(f"A cotação de fechamento é mais informativa: log loss menor em "
              f"{ll_pre - ll_fec:.5f} ({100*(ll_pre-ll_fec)/ll_pre:.2f}%).")
    else:
        print(f"A cotação de fechamento NÃO foi mais informativa nesta amostra "
              f"(diferença de {ll_fec - ll_pre:.5f}).")
else:
    comparacao_fechamento = pd.DataFrame()
    print("Sem colunas de fechamento: comparação não realizada.")

---
## 5. Atributos históricos sem vazamento

Esta é a parte mais delicada do trabalho. **Vazamento** (*data leakage*) é usar,
para prever uma partida, alguma informação que só existiria depois de ela
acontecer. Um modelo com vazamento acerta muito no papel e não serve para nada
na prática.

A regra aqui é dura: **nenhuma informação do próprio jogo entra como atributo**.
As colunas a seguir ficam proibidas na matriz de atributos e só podem ser usadas
para *construir o histórico* das partidas anteriores:

`FTHG, FTAG, FTR, HTHG, HTAG, HTR, HS, AS, HST, AST, HF, AF, HC, AC, HY, AY, HR, AR`

### 5.1 Formato longo

A base vira duas linhas por partida, uma na perspectiva de cada time, com gols
feitos e sofridos, pontos, finalizações (totais e no alvo) feitas e sofridas e
escanteios feitos e sofridos.

In [ ]:
longo = utils.formato_longo(base_historico)
print(f"Formato longo: {len(longo):,} linhas".replace(",", "."),
      f"= {len(base_historico):,} partidas x 2 perspectivas".replace(",", "."))
print("\nMétricas acompanhadas:", ", ".join(utils.METRICAS_LONGO))
print("\nExemplo — as duas linhas geradas por uma mesma partida:")
exemplo_id = base_historico["partida_id"].iloc[-1]
print(longo[longo["partida_id"] == exemplo_id][
    ["data", "Div", "time", "adversario", "mando", "gols_feitos",
     "gols_sofridos", "pontos", "fin_feitas", "esc_feitos"]].to_string(index=False))

### 5.2 Médias móveis com `shift(1)`

Para cada time, ordenado por data, calculamos a média das **últimas 5 e 10
partidas**. O detalhe que garante a ausência de vazamento é a ordem das
operações: o `shift(1)` vem **antes** do `rolling`, de modo que a partida atual
nunca entra no seu próprio atributo.

O histórico é **contínuo entre temporadas** e inclui o aquecimento de 2000/01 a
2004/05 — por isso as primeiras partidas da amostra analisada já chegam com
média móvel cheia.

Também calculamos a **forma por mando**: média das últimas 5 partidas em casa do
mandante e das últimas 5 fora do visitante (gols e pontos).

In [ ]:
tempo_atributos = time.time()
longo = utils.medias_moveis(longo)
print(f"Médias móveis calculadas em {time.time() - tempo_atributos:.1f}s")
print(f"Janelas: {utils.JANELAS}")
print(f"Colunas de média móvel: "
      f"{len(utils.METRICAS_LONGO) * len(utils.JANELAS)} por time")
print()
print("Exemplo — histórico de um time, mostrando o deslocamento de uma partida:")
um_time = longo[longo["time"] == "Arsenal"].sort_values(["data", "partida_id"])
print(um_time[["data", "adversario", "mando", "gols_feitos", "gols_feitos_m5",
               "pontos", "pontos_m5", "jogos_anteriores"]].head(8).to_string(index=False))
print("\nRepare: 'gols_feitos_m5' da linha N é a média das linhas anteriores, "
      "nunca inclui a linha N.")

### 5.3 Elo pré-jogo

O **Elo** resume a força de cada time em um único número que se atualiza a cada
partida. Regras adotadas, todas do enunciado:

* início em **1500**, fator **K = 20**;
* **vantagem de mando de 60 pontos** somada ao mandante no cálculo da
  expectativa;
* na virada de temporada, cada Elo **regride 1/3 em direção à média** (1500),
  para não carregar indefinidamente o desempenho de anos anteriores;
* **promovidos** entram com a média do Elo dos **rebaixados** da mesma liga na
  temporada anterior;
* o Elo é calculado **por liga** e guardamos sempre o valor de **antes** da
  partida.

In [ ]:
elo = utils.calcular_elo(base_historico, k=20.0, vantagem_mando=60.0,
                         elo_inicial=1500.0, fator_regressao=1/3)
print(f"Elo calculado para {len(elo):,} partidas".replace(",", "."))
print()
print(elo[["elo_mandante", "elo_visitante"]].describe().round(1).to_string())
print()
print(f"Partidas com mandante promovido : {int(elo['promovido_mandante'].sum()):,}"
      .replace(",", "."))
print(f"Partidas com visitante promovido: {int(elo['promovido_visitante'].sum()):,}"
      .replace(",", "."))

# Sanidade: os times de maior Elo ao fim da série devem ser os grandes clubes.
ultimo = base_historico[["partida_id", "HomeTeam", "AwayTeam", "Div", "data"]].merge(
    elo, on="partida_id")
recente = ultimo[ultimo["data"] >= ultimo["data"].max() - pd.Timedelta(days=200)]
ranking = pd.concat([
    recente[["HomeTeam", "Div", "elo_mandante"]].rename(
        columns={"HomeTeam": "time", "elo_mandante": "elo"}),
    recente[["AwayTeam", "Div", "elo_visitante"]].rename(
        columns={"AwayTeam": "time", "elo_visitante": "elo"}),
]).groupby(["Div", "time"])["elo"].mean().sort_values(ascending=False)
print("\nMaiores Elo médios na última temporada da base (teste de sanidade):")
print(ranking.head(10).round(1).to_string())

### 5.4 Matriz de atributos

Cada partida recebe os atributos do mandante, os do visitante, a **diferença
(mandante − visitante)** de cada métrica e a **liga em one-hot**. Partidas em
que algum dos dois times tem **menos de 5 jogos anteriores** na base são
descartadas — com histórico curto demais, a média móvel é ruído.

In [ ]:
atributos = utils.montar_matriz_atributos(base_historico, longo, elo)
COLUNAS_ATRIBUTOS = utils.colunas_de_atributos(atributos)

print(f"Matriz de atributos: {atributos.shape[0]:,} partidas x "
      f"{len(COLUNAS_ATRIBUTOS)} atributos".replace(",", "."))
print()
grupos = {
    "médias móveis do mandante": [c for c in COLUNAS_ATRIBUTOS if c.endswith("_mandante") and c.startswith(tuple(utils.METRICAS_LONGO))],
    "médias móveis do visitante": [c for c in COLUNAS_ATRIBUTOS if c.endswith("_visitante") and c.startswith(tuple(utils.METRICAS_LONGO))],
    "diferenças (mandante - visitante)": [c for c in COLUNAS_ATRIBUTOS if c.startswith("dif_")],
    "forma por mando": [c for c in COLUNAS_ATRIBUTOS if c.startswith("forma_")],
    "Elo e promovidos": [c for c in COLUNAS_ATRIBUTOS if "elo" in c or "promovido" in c],
    "liga (one-hot)": [c for c in COLUNAS_ATRIBUTOS if c.startswith("liga_")],
}
for nome, colunas in grupos.items():
    print(f"   {nome:36s}: {len(colunas):3d}")
print(f"   {'TOTAL':36s}: {len(COLUNAS_ATRIBUTOS):3d}")

### 5.5 Testes obrigatórios contra vazamento

Dois testes, ambos com `assert` — se falharem, o notebook para.

1. **Nenhuma coluna proibida** na matriz de atributos.
2. **Recálculo manual**: para uma amostra aleatória de partidas, recalculamos um
   atributo móvel do zero, usando só os jogos anteriores do time, e comparamos
   com o valor da base. Se a partida atual estivesse entrando no próprio
   cálculo, os valores divergiriam.

In [ ]:
# Teste 1: nenhuma coluna do próprio jogo na matriz de atributos.
proibidas_encontradas = [
    coluna for coluna in COLUNAS_ATRIBUTOS
    if any(coluna == proibida or coluna.startswith(proibida + "_")
           for proibida in utils.COLUNAS_PROIBIDAS)]
assert not proibidas_encontradas, f"coluna proibida na matriz: {proibidas_encontradas}"
print(f"TESTE 1 OK — nenhuma das {len(utils.COLUNAS_PROIBIDAS)} colunas proibidas "
      f"aparece entre os {len(COLUNAS_ATRIBUTOS)} atributos.")
print("Colunas proibidas verificadas:", ", ".join(utils.COLUNAS_PROIBIDAS))

In [ ]:
# Teste 2: recálculo manual das médias móveis em amostra aleatória.
print("TESTE 2 — recálculo manual de atributos móveis\n")
verificacoes = []
for metrica, janela in [("gols_feitos", 5), ("pontos", 10),
                        ("fin_alvo_feitas", 5), ("esc_sofridos", 10)]:
    conferencia = utils.verificar_media_movel(
        longo, metrica=metrica, janela=janela, n_amostras=150)
    conferem = int(conferencia["confere"].sum())
    verificacoes.append({"atributo": f"{metrica}_m{janela}",
                         "amostras": len(conferencia), "conferem": conferem})
    print(f"   {metrica}_m{janela:<3d}: {conferem}/{len(conferencia)} conferem")
    assert conferem == len(conferencia), f"divergência em {metrica}_m{janela}"

print("\nExemplo de uma verificação (partida sorteada):")
exemplo_conf = utils.verificar_media_movel(longo, "gols_feitos", 5, n_amostras=6)
print(exemplo_conf.to_string(index=False))
print("\nTESTE 2 OK — todos os valores recalculados batem com a base.")
utils.salvar_tabela(pd.DataFrame(verificacoes), "verificacao_vazamento")

### 5.6 Amostra final da modelagem

In [ ]:
# Junta atributos, alvo e probabilidades do mercado numa única tabela.
alvo = base_historico[["partida_id", "FTR", "HomeTeam", "AwayTeam"]]
dados = atributos.merge(alvo, on="partida_id", how="left")
dados = dados.merge(
    mercado[["partida_id", "prob_H", "prob_D", "prob_A", "favorito", "margem"]],
    on="partida_id", how="left")

registro_filtros = [{"etapa": "matriz de atributos completa", "n": len(dados)}]

# 1) Só a amostra analisada (o aquecimento sai: serviu para construir histórico).
dados = dados[dados["ano_temporada"].between(2005, 2024)]
registro_filtros.append({"etapa": "temporadas 2005/06 a 2024/25", "n": len(dados)})

# 2) Só partidas com probabilidade do mercado (necessária para o baseline).
dados = dados[dados[["prob_H", "prob_D", "prob_A"]].notna().all(axis=1)]
registro_filtros.append({"etapa": "com cotação Bet365", "n": len(dados)})

# 3) Ambos os times com pelo menos 5 jogos anteriores na base.
historico_curto = ((dados["jogos_anteriores_mandante"] < 5)
                   | (dados["jogos_anteriores_visitante"] < 5))
n_historico_curto = int(historico_curto.sum())
dados = dados[~historico_curto]
registro_filtros.append({"etapa": "ambos os times com >= 5 jogos anteriores",
                         "n": len(dados)})

dados = dados.sort_values(["data", "partida_id"], kind="mergesort").reset_index(drop=True)

print("Funil de filtros da modelagem:")
for etapa in registro_filtros:
    print(f"   {etapa['etapa']:44s}: {etapa['n']:>7,}".replace(",", "."))
print(f"\nPartidas descartadas por histórico curto (< 5 jogos): {n_historico_curto}")
utils.salvar_tabela(pd.DataFrame(registro_filtros), "funil_filtros")

print(f"\nDistribuição do alvo (FTR) na amostra:")
print((100 * dados["FTR"].value_counts(normalize=True)[utils.CLASSES]).round(2).to_string())

---
## 6. Modelagem

### 6.1 Divisão temporal

Prever futebol é um problema **temporal**: embaralhar as partidas deixaria o
modelo treinar com jogos posteriores aos que ele tenta prever, o que é outra
forma de vazamento. Por isso o corte é por temporada e **nunca** aleatório.

* **Treino**: 2005/06 a 2021/22
* **Teste**: 2022/23 a 2024/25 — usado **uma única vez**, no fim
* **Validação** (dentro do treino): treina até T−1 e valida em T, para
  T = 2019/20, 2020/21 e 2021/22

In [ ]:
treino = dados[dados["ano_temporada"].isin(utils.ANOS_TREINO)].copy()
teste = dados[dados["ano_temporada"].isin(utils.ANOS_TESTE)].copy()

N_TREINO, N_TESTE = len(treino), len(teste)
print(f"Treino: {N_TREINO:,} partidas ({utils.rotulo_temporada(utils.ANOS_TREINO[0])} "
      f"a {utils.rotulo_temporada(utils.ANOS_TREINO[-1])})".replace(",", "."))
print(f"Teste : {N_TESTE:,} partidas ({utils.rotulo_temporada(utils.ANOS_TESTE[0])} "
      f"a {utils.rotulo_temporada(utils.ANOS_TESTE[-1])})".replace(",", "."))
print(f"Total : {N_TREINO + N_TESTE:,}".replace(",", "."))
assert treino["data"].max() < teste["data"].min(), "sobreposição temporal entre treino e teste"
print(f"\nÚltima partida de treino: {treino['data'].max().date()}")
print(f"Primeira partida de teste: {teste['data'].min().date()}")
print("Conferido: não há sobreposição temporal.")

# Variante A: só desempenho. Variante B: desempenho + probabilidades do mercado.
ATRIBUTOS_A = COLUNAS_ATRIBUTOS
ATRIBUTOS_B = COLUNAS_ATRIBUTOS + ["prob_H", "prob_D", "prob_A"]
VARIANTES = {"A": ATRIBUTOS_A, "B": ATRIBUTOS_B}
print(f"\nVariante A (principal)    : {len(ATRIBUTOS_A)} atributos, sem cotações")
print(f"Variante B (complementar): {len(ATRIBUTOS_B)} atributos, com as "
      f"probabilidades normalizadas da Bet365")

### 6.2 Modelos, pipelines e grades

Todo pré-processamento fica dentro de um `Pipeline` do scikit-learn, o que
garante que a **imputação pela mediana** e a **padronização** sejam ajustadas
só com o treino — nunca com dados de validação ou de teste.

A padronização entra apenas na regressão logística, que é sensível à escala;
árvores e florestas não precisam dela.

As grades são propositalmente pequenas, para manter a execução do notebook
abaixo de 15 minutos numa máquina comum.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def pipeline_arvore(**parametros):
    return Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),
        ("modelo", DecisionTreeClassifier(random_state=utils.SEMENTE, **parametros))])

def pipeline_logistica(**parametros):
    return Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),
        ("padronizacao", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=2000, random_state=utils.SEMENTE,
                                      **parametros))])

def pipeline_floresta(**parametros):
    return Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),
        ("modelo", RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                          random_state=utils.SEMENTE, **parametros))])

MODELOS = {
    "Árvore de decisão": {
        "construtor": pipeline_arvore,
        "grade": [{"max_depth": d, "min_samples_leaf": f}
                  for d in (3, 5, 8, 12) for f in (50, 200)]},
    "Regressão logística": {
        "construtor": pipeline_logistica,
        "grade": [{"C": c} for c in (0.01, 0.1, 1.0, 10.0)]},
    "Floresta aleatória": {
        "construtor": pipeline_floresta,
        "grade": [{"max_depth": d, "min_samples_leaf": f}
                  for d in (10, None) for f in (20, 50)]},
}

for nome, config in MODELOS.items():
    print(f"{nome:22s}: {len(config['grade'])} combinações")
print(f"\nValidação: {len(utils.ANOS_VALIDACAO)} temporadas "
      f"({', '.join(utils.rotulo_temporada(a) for a in utils.ANOS_VALIDACAO)})")
print(f"Total de ajustes por variante: "
      f"{sum(len(c['grade']) for c in MODELOS.values()) * len(utils.ANOS_VALIDACAO)}")

### 6.3 Ajuste de hiperparâmetros por validação temporal

Para cada combinação da grade e cada temporada de validação T, o modelo treina
com tudo **até T−1** e é avaliado em T. A métrica otimizada é o **log loss**,
que avalia a probabilidade inteira e não só a classe escolhida — mais adequado
que a acurácia para comparar com o mercado.

O conjunto de teste **não é tocado** nesta etapa.

In [ ]:
def probabilidades_ordenadas(modelo_ajustado, X):
    # predict_proba devolve as colunas na ordem de classes_ (alfabética: A,D,H).
    # Reordenamos para a ordem fixa H, D, A usada em todo o notebook.
    probabilidades = modelo_ajustado.predict_proba(X)
    indices = [list(modelo_ajustado.classes_).index(c) for c in utils.CLASSES]
    return probabilidades[:, indices]

def validar_combinacao(construtor, parametros, treino, colunas):
    perdas = []
    for ano_validacao in utils.ANOS_VALIDACAO:
        sub_treino = treino[treino["ano_temporada"] < ano_validacao]
        sub_validacao = treino[treino["ano_temporada"] == ano_validacao]
        if sub_treino.empty or sub_validacao.empty:
            continue
        modelo = construtor(**parametros)
        modelo.fit(sub_treino[colunas], sub_treino["FTR"])
        probabilidades = probabilidades_ordenadas(modelo, sub_validacao[colunas])
        perdas.append(utils.log_loss_ordenado(sub_validacao["FTR"], probabilidades))
    return float(np.mean(perdas)), perdas

resultados_validacao = []
melhores_parametros = {}

for variante, colunas in VARIANTES.items():
    for nome, config in MODELOS.items():
        inicio = time.time()
        melhor_perda, melhor = np.inf, None
        for parametros in config["grade"]:
            perda_media, perdas = validar_combinacao(
                config["construtor"], parametros, treino, colunas)
            resultados_validacao.append({
                "variante": variante, "modelo": nome,
                "parametros": str(parametros), "log_loss_medio": perda_media,
                **{f"log_loss_{utils.rotulo_temporada(a)}": p
                   for a, p in zip(utils.ANOS_VALIDACAO, perdas)}})
            if perda_media < melhor_perda:
                melhor_perda, melhor = perda_media, parametros
        melhores_parametros[(variante, nome)] = melhor
        print(f"Variante {variante} | {nome:22s} | melhor log loss "
              f"{melhor_perda:.5f} | {melhor} | {time.time()-inicio:.0f}s")

tabela_validacao = pd.DataFrame(resultados_validacao)
utils.salvar_tabela(tabela_validacao.round(6), "validacao_hiperparametros")

### 6.4 Ajuste final

Escolhidos os hiperparâmetros, cada modelo é reajustado com **todo o treino**
(2005/06 a 2021/22) e aplicado ao teste. A partir daqui o teste não influencia
mais nenhuma decisão.

In [ ]:
modelos_ajustados = {}
probabilidades_teste = {}

for variante, colunas in VARIANTES.items():
    for nome, config in MODELOS.items():
        parametros = melhores_parametros[(variante, nome)]
        modelo = config["construtor"](**parametros)
        inicio = time.time()
        modelo.fit(treino[colunas], treino["FTR"])
        modelos_ajustados[(variante, nome)] = modelo
        probabilidades_teste[(variante, nome)] = probabilidades_ordenadas(
            modelo, teste[colunas])
        print(f"Variante {variante} | {nome:22s} | ajustado em "
              f"{time.time()-inicio:.1f}s | {parametros}")

print(f"\nTempo decorrido até aqui: {(time.time() - INICIO_EXECUCAO)/60:.1f} min")

---
## 7. Avaliação

Todos os modelos e todos os baselines são avaliados **exatamente nas mesmas
partidas de teste**, o que torna a comparação pareada e justa.

### 7.1 Baselines

* **Favorito do mercado**: escolhe o desfecho de maior probabilidade normalizada
  da Bet365. É a referência que interessa ao resumo.
* **Sempre mandante**: aposta sempre na vitória do time da casa. Serve para
  mostrar o quanto a vantagem de jogar em casa já explica sozinha.
* **Frequência das classes no treino**: prevê sempre a distribuição observada no
  treino. É o piso de qualquer modelo probabilístico.

### 7.2 Métricas

* **Acurácia**: proporção de acertos da classe escolhida.
* **Log loss**: penaliza probabilidades confiantes e erradas. Menor é melhor.
* **RPS** (*ranked probability score*): leva em conta a **ordem** H → D → A, e
  penaliza menos quem erra "por pouco" (prever vitória do mandante e sair
  empate) do que quem erra feio (prever mandante e sair visitante). Implementado
  à mão, na ordem fixa H, D, A. Menor é melhor.

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

# Lembrete: o log loss vem de utils.log_loss_ordenado, não do scikit-learn.
# O sklearn ordena "labels" alfabeticamente (A, D, H) e pressupõe essa ordem
# nas colunas de y_pred; aqui a ordem é sempre H, D, A.

y_teste = teste["FTR"].to_numpy()
prob_mercado_teste = teste[["prob_H", "prob_D", "prob_A"]].to_numpy()

# Baseline 1: favorito do mercado.
pred_mercado = np.array(utils.CLASSES)[prob_mercado_teste.argmax(axis=1)]

# Baseline 2: sempre mandante (regra sem probabilidade própria).
prob_sempre_casa = np.tile([1.0, 0.0, 0.0], (len(teste), 1))
pred_sempre_casa = np.full(len(teste), "H")

# Baseline 3: frequência das classes NO TREINO (nunca no teste).
frequencias_treino = (treino["FTR"].value_counts(normalize=True)
                      .reindex(utils.CLASSES).to_numpy())
prob_frequencia = np.tile(frequencias_treino, (len(teste), 1))
pred_frequencia = np.full(len(teste), utils.CLASSES[int(frequencias_treino.argmax())])

print("Frequência das classes no treino (H, D, A):",
      ", ".join(f"{100*f:.2f}%" for f in frequencias_treino))
print("Frequência das classes no teste  (H, D, A):",
      ", ".join(f"{100*(y_teste == c).mean():.2f}%" for c in utils.CLASSES))

In [ ]:
def avaliar(nome, variante, y, probabilidades, predicoes, tem_probabilidade=True):
    acuracia = accuracy_score(y, predicoes)
    perda = (utils.log_loss_ordenado(y, probabilidades)
             if tem_probabilidade else np.nan)
    precisao, revocacao, f1, suporte = precision_recall_fscore_support(
        y, predicoes, labels=utils.CLASSES, zero_division=0)
    registro = {
        "modelo": nome, "variante": variante, "n_teste": len(y),
        "acuracia": acuracia,
        "log_loss": perda,
        "rps": utils.rps(y, probabilidades),
        "empates_previstos_pct": 100 * float(np.mean(predicoes == "D")),
    }
    for i, classe in enumerate(utils.CLASSES):
        registro[f"precisao_{classe}"] = precisao[i]
        registro[f"revocacao_{classe}"] = revocacao[i]
        registro[f"f1_{classe}"] = f1[i]
        registro[f"suporte_{classe}"] = int(suporte[i])
    return registro

linhas_metricas = [
    avaliar("Mercado (favorito Bet365)", "—", y_teste, prob_mercado_teste, pred_mercado),
    avaliar("Sempre mandante", "—", y_teste, prob_sempre_casa, pred_sempre_casa,
            tem_probabilidade=False),
    avaliar("Frequência das classes (treino)", "—", y_teste, prob_frequencia,
            pred_frequencia),
]
predicoes_modelos = {}
for (variante, nome), probabilidades in probabilidades_teste.items():
    predicoes = np.array(utils.CLASSES)[probabilidades.argmax(axis=1)]
    predicoes_modelos[(variante, nome)] = predicoes
    linhas_metricas.append(avaliar(nome, variante, y_teste, probabilidades, predicoes))

metricas = pd.DataFrame(linhas_metricas)
caminho_metricas = utils.salvar_tabela(metricas.round(6), "metricas")

print("Métricas no conjunto de teste (2022/23 a 2024/25):\n")
print(metricas[["modelo", "variante", "acuracia", "log_loss", "rps",
                "empates_previstos_pct"]]
      .assign(acuracia=lambda d: (100*d.acuracia).round(2),
              log_loss=lambda d: d.log_loss.round(5),
              rps=lambda d: d.rps.round(5),
              empates_previstos_pct=lambda d: d.empates_previstos_pct.round(2))
      .to_string(index=False))
print("\nObs.: 'Sempre mandante' é uma regra sem probabilidade própria; o log "
      "loss não se aplica (o RPS usa a previsão degenerada 1/0/0).")

In [ ]:
# Precisão, revocação e F1 por classe — com atenção especial ao empate.
detalhe_classes = metricas[["modelo", "variante"]].copy()
for classe, rotulo in zip(utils.CLASSES, ["mandante", "empate", "visitante"]):
    detalhe_classes[f"precisão_{rotulo}"] = (100*metricas[f"precisao_{classe}"]).round(2)
    detalhe_classes[f"revocação_{rotulo}"] = (100*metricas[f"revocacao_{classe}"]).round(2)
    detalhe_classes[f"F1_{rotulo}"] = (100*metricas[f"f1_{classe}"]).round(2)

print("Desempenho por classe (%):\n")
print(detalhe_classes.to_string(index=False))
print()
print("O EMPATE é o ponto cego de todo mundo. Percentual de empates previstos:")
for _, linha in metricas.iterrows():
    rotulo = f"{linha['modelo']} ({linha['variante']})" if linha["variante"] != "—" else linha["modelo"]
    print(f"   {rotulo:38s}: {linha['empates_previstos_pct']:5.2f}% previstos | "
          f"revocação do empate {100*linha['revocacao_D']:5.2f}%")
print(f"\nNa realidade, {100*(y_teste == 'D').mean():.2f}% das partidas de teste "
      f"terminaram empatadas.")

### 7.3 Matrizes de confusão

As linhas são o resultado real, as colunas o previsto. O que salta à vista é a
coluna do empate: praticamente vazia para todos os modelos e para o próprio
mercado. Prever empate quase nunca é a aposta de maior probabilidade, porque o
empate raramente passa de ~30% mesmo nos jogos mais equilibrados.

In [ ]:
paineis = [("Mercado (favorito Bet365)", pred_mercado)]
paineis += [(nome, predicoes_modelos[("A", nome)]) for nome in MODELOS]

fig, eixos = plt.subplots(1, 4, figsize=(15, 4.1))
mapa = utils.mapa_sequencial()
rotulos = ["Mandante\n(H)", "Empate\n(D)", "Visitante\n(A)"]

for eixo, (titulo, predicoes) in zip(eixos, paineis):
    matriz = confusion_matrix(y_teste, predicoes, labels=utils.CLASSES)
    proporcao = matriz / matriz.sum()
    eixo.imshow(proporcao, cmap=mapa, vmin=0, vmax=proporcao.max())
    eixo.set_xticks(range(3)); eixo.set_xticklabels(rotulos, fontsize=8)
    eixo.set_yticks(range(3)); eixo.set_yticklabels(rotulos, fontsize=8)
    eixo.set_xlabel("Previsto"); eixo.set_ylabel("Real")
    eixo.set_title(titulo, fontsize=10)
    eixo.grid(False)
    limite = proporcao.max() * 0.55
    for i in range(3):
        for j in range(3):
            eixo.text(j, i, f"{matriz[i, j]:,}".replace(",", ".") +
                      f"\n{100*proporcao[i, j]:.1f}%",
                      ha="center", va="center", fontsize=8,
                      color="#ffffff" if proporcao[i, j] > limite else utils.TINTA_PRIMARIA)

fig.suptitle("Matrizes de confusão no teste (variante A) — contagem e % do total",
             fontsize=12, fontweight="semibold")
fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "matrizes_confusao").relative_to(utils.RAIZ))
plt.show()

### 7.4 Comparação com o mercado

Duas ferramentas para decidir se uma diferença de acurácia é real ou ruído:

* **Bootstrap pareado** (2000 reamostragens): reamostra as **mesmas** partidas
  para modelo e mercado e devolve o IC 95% da diferença de acurácia. Se o
  intervalo contém 0, não há evidência de diferença.
* **Teste de McNemar**: olha só as partidas em que os dois discordam. É o teste
  correto para comparar dois classificadores no mesmo conjunto.

In [ ]:
acertos_mercado = (pred_mercado == y_teste).astype(int)
linhas_comparacao = []

for (variante, nome), predicoes in predicoes_modelos.items():
    acertos_modelo = (predicoes == y_teste).astype(int)
    diferenca, inferior, superior = utils.bootstrap_pareado_acuracia(
        acertos_modelo, acertos_mercado, n_reamostras=2000)
    estatistica, p_valor, tabela = utils.teste_mcnemar(acertos_modelo, acertos_mercado)
    supera = bool(inferior > 0 and p_valor < 0.05)
    linhas_comparacao.append({
        "variante": variante, "modelo": nome,
        "acuracia_modelo": acertos_modelo.mean(),
        "acuracia_mercado": acertos_mercado.mean(),
        "diferenca_pp": 100 * diferenca,
        "ic95_inferior_pp": 100 * inferior,
        "ic95_superior_pp": 100 * superior,
        "mcnemar_estatistica": estatistica,
        "mcnemar_p": p_valor,
        "so_modelo_acerta": tabela[0][1],
        "so_mercado_acerta": tabela[1][0],
        "supera_mercado": supera})

comparacao_mercado = pd.DataFrame(linhas_comparacao)
utils.salvar_tabela(comparacao_mercado.round(6), "comparacao_com_mercado")

print("Diferença de acurácia (modelo − mercado), em pontos percentuais:\n")
print(comparacao_mercado.assign(
    acuracia_modelo=lambda d: (100*d.acuracia_modelo).round(2),
    acuracia_mercado=lambda d: (100*d.acuracia_mercado).round(2),
    diferenca_pp=lambda d: d.diferenca_pp.round(2),
    ic95_inferior_pp=lambda d: d.ic95_inferior_pp.round(2),
    ic95_superior_pp=lambda d: d.ic95_superior_pp.round(2),
    mcnemar_p=lambda d: d.mcnemar_p.round(4),
)[["variante", "modelo", "acuracia_modelo", "acuracia_mercado", "diferenca_pp",
   "ic95_inferior_pp", "ic95_superior_pp", "mcnemar_p", "supera_mercado"]]
   .to_string(index=False))

N_SUPERAM_MERCADO = int(comparacao_mercado["supera_mercado"].sum())
print(f"\nModelos que superam o mercado com significância estatística: "
      f"{N_SUPERAM_MERCADO} de {len(comparacao_mercado)}")

### 7.5 Consistência por temporada e por liga

Uma média geral pode esconder que o modelo ganha em uma liga e perde em todas as
outras. Aqui quebramos o teste em cada combinação de **temporada × liga**
(3 temporadas × 5 ligas = 15 recortes) e contamos em quantos deles cada modelo
supera o mercado.

In [ ]:
linhas_consistencia = []
for (variante, nome), predicoes in predicoes_modelos.items():
    probabilidades = probabilidades_teste[(variante, nome)]
    for (ano, liga), indices in teste.groupby(["ano_temporada", "Div"]).groups.items():
        posicoes = teste.index.get_indexer(indices)
        y_recorte = y_teste[posicoes]
        linhas_consistencia.append({
            "variante": variante, "modelo": nome,
            "temporada": utils.rotulo_temporada(ano), "liga": liga,
            "n": len(posicoes),
            "acuracia_modelo": (predicoes[posicoes] == y_recorte).mean(),
            "acuracia_mercado": (pred_mercado[posicoes] == y_recorte).mean(),
            "log_loss_modelo": utils.log_loss_ordenado(
                y_recorte, probabilidades[posicoes]),
            "log_loss_mercado": utils.log_loss_ordenado(
                y_recorte, prob_mercado_teste[posicoes])})

consistencia = pd.DataFrame(linhas_consistencia)
consistencia["modelo_melhor_acuracia"] = (consistencia["acuracia_modelo"]
                                          > consistencia["acuracia_mercado"])
consistencia["modelo_melhor_log_loss"] = (consistencia["log_loss_modelo"]
                                          < consistencia["log_loss_mercado"])
utils.salvar_tabela(consistencia.round(6), "consistencia_temporada_liga")

resumo_consistencia = (consistencia.groupby(["variante", "modelo"])
                       .agg(recortes=("n", "size"),
                            vence_em_acuracia=("modelo_melhor_acuracia", "sum"),
                            vence_em_log_loss=("modelo_melhor_log_loss", "sum"),
                            acuracia_media=("acuracia_modelo", "mean"))
                       .reset_index())
print("Em quantos dos 15 recortes (temporada x liga) cada modelo supera o mercado:\n")
print(resumo_consistencia.assign(
    acuracia_media=lambda d: (100*d.acuracia_media).round(2)).to_string(index=False))

### 7.6 Importância dos atributos

Duas leituras diferentes e complementares:

* **Importância por permutação** (floresta aleatória, medida **no teste**):
  embaralha uma coluna por vez e mede o quanto a acurácia cai. Mede o que o
  modelo de fato usa.
* **Coeficientes** (regressão logística): o sinal diz a direção do efeito. Como
  os atributos são padronizados, as magnitudes são comparáveis entre si.

In [ ]:
from sklearn.inspection import permutation_importance

inicio = time.time()
floresta_A = modelos_ajustados[("A", "Floresta aleatória")]
importancia = permutation_importance(
    floresta_A, teste[ATRIBUTOS_A], teste["FTR"],
    n_repeats=5, random_state=utils.SEMENTE, scoring="accuracy", n_jobs=-1)

importancia_floresta = pd.DataFrame({
    "atributo": ATRIBUTOS_A,
    "queda_media_acuracia": importancia.importances_mean,
    "desvio": importancia.importances_std,
}).sort_values("queda_media_acuracia", ascending=False).reset_index(drop=True)
utils.salvar_tabela(importancia_floresta.round(6), "importancia_floresta")
print(f"Importância por permutação calculada em {time.time()-inicio:.0f}s\n")
print(importancia_floresta.head(15).round(5).to_string(index=False))

In [ ]:
logistica_A = modelos_ajustados[("A", "Regressão logística")]
coeficientes = logistica_A.named_steps["modelo"].coef_
classes_modelo = list(logistica_A.named_steps["modelo"].classes_)

coeficientes_lr = pd.DataFrame(coeficientes.T, index=ATRIBUTOS_A,
                               columns=[f"coef_{c}" for c in classes_modelo])
coeficientes_lr["magnitude_maxima"] = coeficientes_lr.abs().max(axis=1)
coeficientes_lr = coeficientes_lr.sort_values("magnitude_maxima", ascending=False)
utils.salvar_tabela(coeficientes_lr.reset_index(names="atributo").round(6),
                    "coeficientes_logistica")
print("Regressão logística — 15 atributos de maior magnitude:\n")
print(coeficientes_lr.head(15).round(4).to_string())

In [ ]:
fig, (eixo_esq, eixo_dir) = plt.subplots(1, 2, figsize=(14, 6))

topo_floresta = importancia_floresta.head(15).iloc[::-1]
eixo_esq.barh(topo_floresta["atributo"], topo_floresta["queda_media_acuracia"],
              xerr=topo_floresta["desvio"], color=utils.CORES_SERIES[0],
              error_kw={"ecolor": utils.TINTA_SECUNDARIA, "elinewidth": 1})
eixo_esq.set_title("Floresta aleatória — importância por permutação (teste)", fontsize=11)
eixo_esq.set_xlabel("Queda média da acurácia ao embaralhar o atributo")
eixo_esq.tick_params(axis="y", labelsize=8)

topo_lr = coeficientes_lr.head(15).iloc[::-1]
coluna_h = f"coef_H" if "coef_H" in topo_lr.columns else topo_lr.columns[0]
# Coeficiente da classe H: sinal positivo puxa para vitória do mandante.
# Cor divergente (azul <-> vermelho) porque o que importa aqui é o SINAL.
cores_lr = [utils.CORES_SERIES[0] if v >= 0 else "#e34948" for v in topo_lr[coluna_h]]
eixo_dir.barh(topo_lr.index, topo_lr[coluna_h], color=cores_lr)
eixo_dir.axvline(0, color=utils.TINTA_SUAVE, linewidth=1.0)
eixo_dir.set_title("Regressão logística — coeficientes da classe H\n"
                   "(azul: favorece o mandante; vermelho: favorece o visitante)",
                   fontsize=11)
eixo_dir.set_xlabel("Coeficiente (atributos padronizados)")
eixo_dir.tick_params(axis="y", labelsize=8)

fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "importancia_atributos").relative_to(utils.RAIZ))
plt.show()

In [ ]:
# Acurácia por temporada de teste — visão de consistência.
fig, eixo = plt.subplots(figsize=(9.5, 5))

recortes_A = consistencia[consistencia["variante"] == "A"]
por_temporada = (recortes_A.groupby(["modelo", "temporada"])
                 .apply(lambda d: np.average(d["acuracia_modelo"], weights=d["n"]),
                        include_groups=False)
                 .reset_index(name="acuracia"))
mercado_temporada = (recortes_A[recortes_A["modelo"] == list(MODELOS)[0]]
                     .groupby("temporada")
                     .apply(lambda d: np.average(d["acuracia_mercado"], weights=d["n"]),
                            include_groups=False))

temporadas = sorted(por_temporada["temporada"].unique())
posicao = np.arange(len(temporadas))
largura = 0.26
for i, nome in enumerate(MODELOS):
    valores = (por_temporada[por_temporada["modelo"] == nome]
               .set_index("temporada").loc[temporadas, "acuracia"].to_numpy())
    eixo.bar(posicao + (i - 1) * (largura + 0.012), 100 * valores, largura,
             label=nome, color=utils.CORES_SERIES[i])

eixo.plot(posicao, 100 * mercado_temporada.loc[temporadas].to_numpy(),
          "o-", color=utils.TINTA_PRIMARIA, linewidth=1.8, markersize=6,
          label="Mercado (favorito Bet365)", zorder=5)

eixo.set_title("Acurácia por temporada de teste (variante A)")
eixo.set_xlabel("Temporada")
eixo.set_ylabel("Acurácia (%)")
eixo.set_xticks(posicao); eixo.set_xticklabels(temporadas)
eixo.set_ylim(0, 65)
eixo.legend(ncol=2)
fig.tight_layout()
print("Figura salva em:", utils.salvar_figura(fig, "acuracia_por_temporada").relative_to(utils.RAIZ))
plt.show()

---
## 8. Números para o resumo

Esta seção fecha o trabalho: confronta cada afirmação do resumo com o que os
dados mostraram, monta a frase de resultados já no formato brasileiro (vírgula
decimal, uma casa) e grava tudo em `outputs/resultados_resumo.txt`.

**Nenhum número desta seção é digitado à mão.** Todos vêm das variáveis
calculadas acima. Onde o dado não confirma o texto do resumo, a seção diz isso
explicitamente.

In [ ]:
# ---------------------------------------------------------------------------
# (1) Tamanhos da amostra
# ---------------------------------------------------------------------------
N_MODELAGEM = N_TREINO + N_TESTE

print("=" * 78)
print("1. TAMANHO DA AMOSTRA")
print("=" * 78)
print(f"N de partidas das ligas principais em 2005/06-2024/25 "
      f"(após limpeza)      : {N_AMOSTRA:,}".replace(",", "."))
print(f"N usado na modelagem (após exigir >= 5 jogos anteriores de cada time): "
      f"{N_MODELAGEM:,}".replace(",", "."))
print(f"   treino (2005/06 a 2021/22): {N_TREINO:,}".replace(",", "."))
print(f"   teste  (2022/23 a 2024/25): {N_TESTE:,}".replace(",", "."))
print()
print(f"O resumo fala em 'cerca de quarenta mil partidas'. O valor apurado é "
      f"{N_AMOSTRA:,}".replace(",", ".") + ".")

In [ ]:
# ---------------------------------------------------------------------------
# (2) Alarme de vazamento
# ---------------------------------------------------------------------------
LIMITE_VAZAMENTO = 0.60
acuracias_modelos = {(v, n): (p.argmax(axis=1) ==
                              np.array([utils.CLASSES.index(y) for y in y_teste])).mean()
                     for (v, n), p in probabilidades_teste.items()}
acuracia_maxima = max(acuracias_modelos.values())
modelo_mais_alto = max(acuracias_modelos, key=acuracias_modelos.get)
ALARME_VAZAMENTO = bool(acuracia_maxima > LIMITE_VAZAMENTO)

print("=" * 78)
print("2. VERIFICAÇÃO DE VAZAMENTO")
print("=" * 78)
print(f"Maior acurácia observada: {100*acuracia_maxima:.2f}% "
      f"({modelo_mais_alto[1]}, variante {modelo_mais_alto[0]})")
print(f"Limite de alarme        : {100*LIMITE_VAZAMENTO:.0f}%")
if ALARME_VAZAMENTO:
    print()
    print("#" * 78)
    print("#  ALARME DE VAZAMENTO                                                    #")
    print("#  Algum modelo passou de 60% de acurácia. Para este problema, isso não   #")
    print("#  é plausível: revise a construção dos atributos antes de usar qualquer  #")
    print("#  número. A FRASE DE RESULTADOS NÃO SERÁ GERADA.                         #")
    print("#" * 78)
else:
    print("OK — nenhum modelo passou do limite. Os números são plausíveis para o "
          "problema.")

In [ ]:
# ---------------------------------------------------------------------------
# (3) A frase de resultados (variante A, a que vai para o resumo)
# ---------------------------------------------------------------------------
acuracia_mercado = float((pred_mercado == y_teste).mean())
acuracia_arvore = float(acuracias_modelos[("A", "Árvore de decisão")])
acuracia_logistica = float(acuracias_modelos[("A", "Regressão logística")])
acuracia_floresta = float(acuracias_modelos[("A", "Floresta aleatória")])

FRASE_RESULTADOS = (
    "Na previsão do resultado das partidas, a árvore de decisão alcançou "
    f"{utils.formatar_br(100*acuracia_arvore, 1)}% de acerto, a regressão "
    f"logística {utils.formatar_br(100*acuracia_logistica, 1)}% e a floresta "
    f"aleatória {utils.formatar_br(100*acuracia_floresta, 1)}%, contra "
    f"{utils.formatar_br(100*acuracia_mercado, 1)}% obtidos ao se adotar apenas "
    "o favorito apontado pelas cotações.")

print("=" * 78)
print("3. FRASE DE RESULTADOS (variante A)")
print("=" * 78)
if ALARME_VAZAMENTO:
    FRASE_RESULTADOS = ("[NÃO GERADA — alarme de vazamento acionado: algum modelo "
                        "passou de 60% de acurácia.]")
    print(FRASE_RESULTADOS)
else:
    print(FRASE_RESULTADOS)

In [ ]:
# ---------------------------------------------------------------------------
# (4) Afirmação 1: a margem varia entre operadores e entre divisões?
# ---------------------------------------------------------------------------
# Critérios declarados ANTES de olhar o resultado:
#   operadores - amplitude >= 0,5 ponto percentual entre a menor e a maior
#                margem média, na janela em que todos coexistem;
#   divisões   - a diferença entre 2a e 1a divisão tem o MESMO sinal em pelo
#                menos 4 dos 5 países (é o que torna a variação "consistente").
LIMITE_AMPLITUDE_OPERADORES = 0.5
MINIMO_PAISES_MESMO_SINAL = 4

diferencas_divisao = tabela_paises["diferença (2ª - 1ª)"]
paises_2a_maior = int((diferencas_divisao > 0).sum())
paises_2a_menor = int((diferencas_divisao < 0).sum())
paises_mesmo_sinal = max(paises_2a_maior, paises_2a_menor)

margem_varia_operadores = bool(amplitude_operadores >= LIMITE_AMPLITUDE_OPERADORES)
margem_varia_divisoes = bool(paises_mesmo_sinal >= MINIMO_PAISES_MESMO_SINAL)
AFIRMACAO_1 = bool(margem_varia_operadores and margem_varia_divisoes)

print("=" * 78)
print("4. AFIRMAÇÃO 1 — 'a margem varia de maneira consistente entre")
print("   operadores e divisões'")
print("=" * 78)
print(f"Entre OPERADORES (janela {utils.rotulo_temporada(primeiro_ano_comum)}–"
      f"{utils.rotulo_temporada(ultimo_ano_comum)}):")
print(f"   menor margem média : {comparacao['média'].idxmin()} "
      f"({utils.formatar_br(comparacao['média'].min(), 2)}%)")
print(f"   maior margem média : {comparacao['média'].idxmax()} "
      f"({utils.formatar_br(comparacao['média'].max(), 2)}%)")
print(f"   amplitude          : {utils.formatar_br(amplitude_operadores, 2)} p.p. "
      f"(critério: >= {utils.formatar_br(LIMITE_AMPLITUDE_OPERADORES, 1)}) -> "
      f"{'varia' if margem_varia_operadores else 'NÃO varia'}")
print()
print("Entre DIVISÕES (Bet365, 2005/06–2024/25):")
for pais, diferenca in diferencas_divisao.items():
    print(f"   {pais:12s}: 1ª {utils.formatar_br(tabela_paises.loc[pais, '1ª divisão'], 2)}%"
          f" | 2ª {utils.formatar_br(tabela_paises.loc[pais, '2ª divisão'], 2)}%"
          f" | diferença {utils.formatar_br(diferenca, 2)} p.p.")
print(f"   países com 2ª divisão de margem MAIOR: {paises_2a_maior} de 5")
print(f"   países com 2ª divisão de margem MENOR: {paises_2a_menor} de 5")
print(f"   mesmo sinal em {paises_mesmo_sinal} de 5 (critério: >= "
      f"{MINIMO_PAISES_MESMO_SINAL}) -> "
      f"{'consistente' if margem_varia_divisoes else 'NÃO consistente'}")
print()
print(f">>> AFIRMAÇÃO 1: {'CONFIRMADA' if AFIRMACAO_1 else 'NÃO CONFIRMADA'}")

In [ ]:
# ---------------------------------------------------------------------------
# (5) Afirmação 2: os desfechos improváveis são superestimados?
# ---------------------------------------------------------------------------
# Critério declarado: as duas leituras têm de apontar na mesma direção —
#   (a) ao menos uma faixa de baixa probabilidade (< 30%) com IC 95% de Wilson
#       inteiramente ABAIXO da probabilidade prevista; e
#   (b) inclinação da regressão logística significativamente MAIOR que 1.
AFIRMACAO_2 = bool(VIES_CONFIRMADO_FAIXAS and VIES_CONFIRMADO_REGRESSAO)

print("=" * 78)
print("5. AFIRMAÇÃO 2 — 'as probabilidades atribuídas aos desfechos menos")
print("   prováveis tendem a ser superestimadas'")
print("=" * 78)
print("(a) Leitura pelas faixas de baixa probabilidade (< 30%):")
for _, linha in faixas_baixas.iterrows():
    prevista, observada = 100*linha["prob_media_prevista"], 100*linha["frequencia_observada"]
    inferior, superior = 100*linha["ic95_inferior"], 100*linha["ic95_superior"]
    marca = "SUPERESTIMADA" if superior < prevista else "dentro do IC"
    print(f"    {linha['faixa']:>14s}: prevista {utils.formatar_br(prevista, 2)}% | "
          f"observada {utils.formatar_br(observada, 2)}% "
          f"[{utils.formatar_br(inferior, 2)}; {utils.formatar_br(superior, 2)}] -> {marca}")
print(f"    faixas superestimadas: {n_superestimadas} de {len(faixas_baixas)} -> "
      f"{'confirma' if VIES_CONFIRMADO_FAIXAS else 'NÃO confirma'}")
print()
print("(b) Regressão logística sobre o logit da probabilidade normalizada:")
print(f"    inclinação {utils.formatar_br(inclinacao, 4)} "
      f"IC 95% [{utils.formatar_br(ic_inclinacao[0], 4)}; "
      f"{utils.formatar_br(ic_inclinacao[1], 4)}]")
print(f"    {veredicto_regressao} -> "
      f"{'confirma' if VIES_CONFIRMADO_REGRESSAO else 'NÃO confirma'}")
print()
print(f">>> AFIRMAÇÃO 2: {'CONFIRMADA' if AFIRMACAO_2 else 'NÃO CONFIRMADA'}")
if not AFIRMACAO_2:
    print()
    print("    Observação importante: o teste foi feito com as probabilidades")
    print("    NORMALIZADAS (com a margem removida). É uma escolha conservadora —")
    print("    parte do que costuma ser lido como viés favorito-azarão vem da")
    print("    própria margem, que incide mais pesadamente sobre as cotações altas.")

In [ ]:
# ---------------------------------------------------------------------------
# (6) Sugestão de frase de conclusão, decidida pelos dados
# ---------------------------------------------------------------------------
LIMITE_DISTANCIA_PP = 3.0
comparacao_A = comparacao_mercado[comparacao_mercado["variante"] == "A"]
algum_supera = bool(comparacao_A["supera_mercado"].any())
melhor_acuracia_A = float(comparacao_A["acuracia_modelo"].max())
distancia_pp = 100 * (acuracia_mercado - melhor_acuracia_A)

# Log loss: o mercado também é melhor aqui?
log_loss_mercado = float(metricas.loc[metricas["modelo"] == "Mercado (favorito Bet365)",
                                      "log_loss"].iloc[0])
log_loss_modelos_A = metricas[(metricas["variante"] == "A")]["log_loss"]
melhor_log_loss_A = float(log_loss_modelos_A.min())
mercado_melhor_log_loss = bool(log_loss_mercado < melhor_log_loss_A)

print("=" * 78)
print("6. SUGESTÃO DE FRASE DE CONCLUSÃO")
print("=" * 78)
print(f"Acurácia do mercado          : {utils.formatar_br(100*acuracia_mercado, 2)}%")
print(f"Melhor modelo (variante A)   : {utils.formatar_br(100*melhor_acuracia_A, 2)}%")
print(f"Distância                    : {utils.formatar_br(distancia_pp, 2)} p.p. "
      f"(limite: {utils.formatar_br(LIMITE_DISTANCIA_PP, 1)})")
print(f"Log loss do mercado          : {log_loss_mercado:.5f}")
print(f"Melhor log loss (variante A) : {melhor_log_loss_A:.5f}")
print(f"Mercado melhor no log loss   : {mercado_melhor_log_loss}")
print(f"Algum modelo supera o mercado com significância: {algum_supera}")
print()

if algum_supera:
    FRASE_CONCLUSAO = (
        "ALERTA: algum modelo superou o mercado com significância estatística. "
        "Antes de aceitar esse resultado, revise a construção dos atributos em "
        "busca de vazamento — superar o mercado de apostas de forma consistente "
        "é um resultado extraordinário e exige evidência à altura.")
    print("!" * 78)
    print(FRASE_CONCLUSAO)
    print("!" * 78)
elif distancia_pp <= LIMITE_DISTANCIA_PP:
    FRASE_CONCLUSAO = ("Conclui-se que os modelos supervisionados aproximam-se do "
                       "desempenho do mercado, mas não o superam de forma consistente.")
    print("Distância de até 3 pontos percentuais e nenhum modelo supera o mercado")
    print("com significância -> MANTER a conclusão provisória do resumo:")
    print()
    print(f"   \"{FRASE_CONCLUSAO}\"")
else:
    FRASE_CONCLUSAO = ("Conclui-se que os modelos supervisionados ficam abaixo do "
                       "desempenho do mercado.")
    print("Distância maior que 3 pontos percentuais -> SUGERIR nova conclusão:")
    print()
    print(f"   \"{FRASE_CONCLUSAO}\"")

In [ ]:
# ---------------------------------------------------------------------------
# (7) Gravação de outputs/resultados_resumo.txt
# ---------------------------------------------------------------------------
def pct(valor, casas=1):
    return utils.formatar_br(100 * valor, casas) + "%"

linhas = []
adicionar = linhas.append

adicionar("=" * 78)
adicionar("NÚMEROS PARA O RESUMO — JECET 2026")
adicionar("Eficiência das cotações de apostas esportivas")
adicionar("Mineração de Dados — ADS, IFSP Câmpus Jacareí")
adicionar("=" * 78)
adicionar(f"Gerado pela execução do notebook em "
          f"{pd.Timestamp.now().strftime('%d/%m/%Y %H:%M')}")
adicionar(f"Fonte dos dados: Football-Data.co.uk "
          f"({'acesso direto' if relatorio_coleta.attrs['oficial_ok'] else 'via espelho público — ver Decisões e limitações'})")
adicionar("")

adicionar("-" * 78)
adicionar("1. AMOSTRA")
adicionar("-" * 78)
adicionar(f"Ligas principais: {', '.join(utils.LIGAS_PRINCIPAIS)} "
          f"({', '.join(utils.NOME_LIGA[l] for l in utils.LIGAS_PRINCIPAIS)})")
adicionar(f"Período analisado: 2005/06 a 2024/25 (20 temporadas)")
adicionar(f"Aquecimento (fora da amostra): 2000/01 a 2004/05")
adicionar("")
adicionar(f"N de partidas analisadas (após limpeza) ....... {N_AMOSTRA:,}".replace(",", "."))
adicionar(f"N usado na modelagem ......................... {N_MODELAGEM:,}".replace(",", "."))
adicionar(f"   N de treino (2005/06 a 2021/22) ........... {N_TREINO:,}".replace(",", "."))
adicionar(f"   N de teste  (2022/23 a 2024/25) ........... {N_TESTE:,}".replace(",", "."))
adicionar("")
adicionar(f"O resumo diz 'cerca de quarenta mil partidas'; o valor apurado é "
          f"{N_AMOSTRA:,}.".replace(",", "."))
adicionar("")

adicionar("-" * 78)
adicionar("2. FRASE DE RESULTADOS (pronta para o resumo — variante A)")
adicionar("-" * 78)
adicionar(FRASE_RESULTADOS)
adicionar("")
adicionar("Métricas completas da variante A no conjunto de teste:")
for _, linha in metricas[metricas["variante"] == "A"].iterrows():
    adicionar(f"   {linha['modelo']:22s} | acurácia {pct(linha['acuracia'], 2):>7s} | "
              f"log loss {utils.formatar_br(linha['log_loss'], 5)} | "
              f"RPS {utils.formatar_br(linha['rps'], 5)}")
linha_mercado = metricas[metricas["modelo"] == "Mercado (favorito Bet365)"].iloc[0]
adicionar(f"   {'Mercado (Bet365)':22s} | acurácia {pct(linha_mercado['acuracia'], 2):>7s} | "
          f"log loss {utils.formatar_br(linha_mercado['log_loss'], 5)} | "
          f"RPS {utils.formatar_br(linha_mercado['rps'], 5)}")
adicionar("")
adicionar("Variante B (complementar — atributos + probabilidades da Bet365):")
for _, linha in metricas[metricas["variante"] == "B"].iterrows():
    adicionar(f"   {linha['modelo']:22s} | acurácia {pct(linha['acuracia'], 2):>7s} | "
              f"log loss {utils.formatar_br(linha['log_loss'], 5)} | "
              f"RPS {utils.formatar_br(linha['rps'], 5)}")
adicionar("")

adicionar("-" * 78)
adicionar("3. AFIRMAÇÕES DE 'RESULTADOS INICIAIS'")
adicionar("-" * 78)
adicionar(f"[{'CONFIRMADA' if AFIRMACAO_1 else 'NÃO CONFIRMADA'}] "
          "A margem embutida nas cotações varia de maneira consistente entre")
adicionar("               operadores e divisões.")
adicionar(f"   Entre operadores ({utils.rotulo_temporada(primeiro_ano_comum)}–"
          f"{utils.rotulo_temporada(ultimo_ano_comum)}): de "
          f"{utils.formatar_br(comparacao['média'].min(), 2)}% "
          f"({comparacao['média'].idxmin()}) a "
          f"{utils.formatar_br(comparacao['média'].max(), 2)}% "
          f"({comparacao['média'].idxmax()}); amplitude "
          f"{utils.formatar_br(amplitude_operadores, 2)} p.p.")
adicionar(f"   Entre divisões (Bet365): a 2ª divisão tem margem maior em "
          f"{paises_2a_maior} dos 5 países;")
adicionar(f"      diferença média (2ª − 1ª) = "
          f"{utils.formatar_br(diferencas_divisao.mean(), 2)} p.p.")
for pais, diferenca in diferencas_divisao.items():
    adicionar(f"      {pais:12s}: 1ª {utils.formatar_br(tabela_paises.loc[pais, '1ª divisão'], 2)}%"
              f" | 2ª {utils.formatar_br(tabela_paises.loc[pais, '2ª divisão'], 2)}%"
              f" | {utils.formatar_br(diferenca, 2):>6s} p.p.")
adicionar("")
adicionar(f"[{'CONFIRMADA' if AFIRMACAO_2 else 'NÃO CONFIRMADA'}] "
          "As probabilidades atribuídas aos desfechos menos prováveis")
adicionar("               tendem a ser superestimadas.")
adicionar(f"   (a) Faixas de baixa probabilidade (< 30%) com superestimação "
          f"significativa: {n_superestimadas} de {len(faixas_baixas)}")
for _, linha in faixas_baixas.iterrows():
    adicionar(f"       {linha['faixa']:>14s}: prevista "
              f"{utils.formatar_br(100*linha['prob_media_prevista'], 2)}% | observada "
              f"{utils.formatar_br(100*linha['frequencia_observada'], 2)}% "
              f"[{utils.formatar_br(100*linha['ic95_inferior'], 2)}; "
              f"{utils.formatar_br(100*linha['ic95_superior'], 2)}]")
adicionar(f"   (b) Inclinação da regressão logística sobre o logit: "
          f"{utils.formatar_br(inclinacao, 4)} "
          f"IC 95% [{utils.formatar_br(ic_inclinacao[0], 4)}; "
          f"{utils.formatar_br(ic_inclinacao[1], 4)}] "
          f"({'> 1 -> viés' if VIES_CONFIRMADO_REGRESSAO else 'não indica o viés'})")
adicionar(f"   Retorno médio por unidade apostada, por faixa de cotação (Bet365):")
for _, linha in retorno.iterrows():
    adicionar(f"       {str(linha['faixa_cotacao']):>12s}: "
              f"{utils.formatar_br(100*linha['retorno_medio'], 2):>7s}% "
              f"[{utils.formatar_br(100*linha['ic95_inferior'], 2)}; "
              f"{utils.formatar_br(100*linha['ic95_superior'], 2)}] "
              f"(n = {int(linha['n_apostas']):,})".replace(",", "."))
adicionar("")

adicionar("-" * 78)
adicionar("4. COMPARAÇÃO COM O MERCADO (teste pareado nas mesmas partidas)")
adicionar("-" * 78)
for _, linha in comparacao_mercado.iterrows():
    adicionar(f"   [{linha['variante']}] {linha['modelo']:22s} | "
              f"diferença {utils.formatar_br(linha['diferenca_pp'], 2):>6s} p.p. | "
              f"IC 95% [{utils.formatar_br(linha['ic95_inferior_pp'], 2)}; "
              f"{utils.formatar_br(linha['ic95_superior_pp'], 2)}] | "
              f"McNemar p = {utils.formatar_br(linha['mcnemar_p'], 4)} | "
              f"{'SUPERA' if linha['supera_mercado'] else 'não supera'}")
adicionar("")
adicionar("   Consistência (em quantos dos 15 recortes temporada x liga o modelo")
adicionar("   supera o mercado):")
for _, linha in resumo_consistencia.iterrows():
    adicionar(f"      [{linha['variante']}] {linha['modelo']:22s} | "
              f"acurácia: {int(linha['vence_em_acuracia'])}/{int(linha['recortes'])} | "
              f"log loss: {int(linha['vence_em_log_loss'])}/{int(linha['recortes'])}")
adicionar("")

adicionar("-" * 78)
adicionar("5. SUGESTÃO DE FRASE DE CONCLUSÃO")
adicionar("-" * 78)
adicionar(FRASE_CONCLUSAO)
adicionar("")
adicionar(f"   Base da decisão: distância do melhor modelo para o mercado = "
          f"{utils.formatar_br(distancia_pp, 2)} p.p. "
          f"(limite adotado: {utils.formatar_br(LIMITE_DISTANCIA_PP, 1)} p.p.);")
adicionar(f"   modelos que superam o mercado com significância: {N_SUPERAM_MERCADO};")
adicionar(f"   o mercado também é melhor no log loss: "
          f"{'sim' if mercado_melhor_log_loss else 'não'}.")
adicionar("")

adicionar("-" * 78)
adicionar("6. VERIFICAÇÃO DE VAZAMENTO")
adicionar("-" * 78)
adicionar(f"   Maior acurácia observada: {pct(acuracia_maxima, 2)} "
          f"({modelo_mais_alto[1]}, variante {modelo_mais_alto[0]}); "
          f"limite de alarme: {pct(LIMITE_VAZAMENTO, 0)}")
if ALARME_VAZAMENTO:
    adicionar("   *** ALARME ACIONADO — revisar a construção dos atributos. ***")
else:
    adicionar("   Sem alarme. Testes de vazamento (colunas proibidas e recálculo")
    adicionar("   manual das médias móveis) passaram.")
adicionar("")
adicionar("=" * 78)
adicionar(f"Tempo total de execução do notebook: "
          f"{(time.time() - INICIO_EXECUCAO)/60:.1f} minutos")
adicionar("=" * 78)

texto_resumo = "\n".join(linhas)
caminho_resumo = utils.DIR_SAIDAS / "resultados_resumo.txt"
caminho_resumo.write_text(texto_resumo, encoding="utf-8")

print(texto_resumo)
print()
print("Arquivo gravado em:", caminho_resumo.relative_to(utils.RAIZ))

In [ ]:
# Lista final dos arquivos gerados.
print("ARQUIVOS GERADOS\n")
for pasta in (utils.DIR_FIGURAS, utils.DIR_TABELAS):
    arquivos = sorted(pasta.glob("*"))
    print(f"{pasta.relative_to(utils.RAIZ)}/  ({len(arquivos)} arquivos)")
    for arquivo in arquivos:
        print(f"   {arquivo.name:44s} {arquivo.stat().st_size/1024:8.1f} KB")
    print()
print(f"{(utils.DIR_SAIDAS / 'resultados_resumo.txt').relative_to(utils.RAIZ)}")
print(f"\nTempo total de execução: {(time.time() - INICIO_EXECUCAO)/60:.1f} minutos")

---
## 9. Decisões e limitações

Esta seção registra as escolhas feitas ao longo do trabalho, inclusive as que o
enunciado não cobria. O critério, nesses casos, foi sempre a opção mais simples
e defensável.

### Fonte dos dados

* **A fonte oficial (`football-data.co.uk`) está bloqueada no ambiente em que
  este notebook foi executado** — a política de rede do ambiente recusa a
  conexão com esse domínio. O notebook tenta a fonte oficial primeiro e, quando
  ela falha, cai para um **espelho público** que republica os mesmos arquivos
  CSV, com as colunas originais preservadas. A célula de coleta registra a
  origem de cada arquivo e emite aviso quando o espelho é usado.
* A integridade da base foi conferida contra fatos conhecidos das competições:
  306 partidas por temporada na Bundesliga (18 clubes), 380 nas demais ligas de
  20 clubes, 306 na Serie A até 2004/05, 279 na Ligue 1 em 2019/20 (temporada
  interrompida pela pandemia) e 306 a partir de 2023/24 (redução para 18
  clubes). Todos batem.
* **Quem rodar o notebook numa rede sem esse bloqueio usará a fonte oficial
  automaticamente**, sem alterar nenhuma linha de código.

### Tratamento das equipes

* O dicionário de renomeações saiu **vazio**: o Football-Data.co.uk já usa
  nomes curtos consistentes entre temporadas, e não há colisão de normalização
  em nenhuma das cinco ligas principais. A limpeza de espaços continua sendo
  aplicada sempre.
* Os pares de nomes parecidos que a busca automática aponta (`Piacenza`/
  `Vicenza`, `Le Mans`/`Lens`, `Ajaccio`/`Ajaccio GFCO`) são **clubes
  diferentes** e foram mantidos separados.

### Elo

* A regressão de 1/3 na virada de temporada é feita **em direção a 1500**, a
  média de referência da liga, e não à média corrente dos clubes presentes — é
  a definição mais simples e não depende de quem entrou ou saiu naquele ano.
* A ordem das operações na virada de temporada é: primeiro a regressão de todos
  os times conhecidos, depois o cálculo da média dos rebaixados (já regredidos)
  e a atribuição desse valor aos promovidos. Assim promovidos e remanescentes
  ficam na mesma escala.
* Um time que volta à liga **depois de vários anos fora** é tratado como
  promovido e recebe a média dos rebaixados, em vez de recuperar seu Elo antigo.
  É defensável: um Elo de cinco anos atrás diz pouco sobre o elenco atual.
* A vantagem de mando ficou fixa em **60 pontos**, sem ajuste. O enunciado
  permitia calibrá-la no treino; como o Elo é apenas um entre 68 atributos, o
  ganho esperado não justificava o custo.

### Colunas indisponíveis

* As **estatísticas de jogo** (finalizações, escanteios) só ficam completas a
  partir de 2005/06. Nas temporadas de aquecimento a cobertura é parcial, o que
  afeta o início de algumas médias móveis — os valores faltantes são tratados
  pela imputação de mediana, ajustada só no treino.
* Algumas ligas têm lacunas próprias: a Bundesliga não traz finalizações no
  alvo em parte do período e a Ligue 1 tem escanteios faltando em algumas
  temporadas. A tabela `outputs/tabelas/cobertura.csv` documenta tudo, liga por
  liga e temporada por temporada.
* As **cotações de fechamento** (sufixo `C`) só existem de 2019/20 em diante, e
  por isso a comparação pré-jogo × fechamento se restringe a essas temporadas.
* Os operadores entram e saem da base ao longo dos anos. Cada um só entra na
  análise de margem nas temporadas em que cobre pelo menos 80% das partidas.
* A **média de mercado** muda de nome (`BbAv*` até 2018/19, `Avg*` depois); o
  notebook trata as duas como o mesmo operador.

### Partidas descartadas

* Partidas sem resultado ou sem cotação da Bet365 saem da amostra, com a
  contagem registrada na seção 3.4.
* Partidas em que algum dos dois times tinha **menos de 5 jogos anteriores** na
  base também saem — a contagem está no funil de filtros da seção 5.6. O
  aquecimento de 2000/01 a 2004/05 reduz bastante esse descarte, porque os
  times já chegam a 2005/06 com histórico.
* As temporadas de aquecimento **não** passam pelo filtro de cotação: elas
  servem só para construir histórico de desempenho, e a Bet365 só aparece na
  base a partir de 2002/03.

### Escolhas estatísticas

* O intervalo de confiança das proporções é o de **Wilson**, mais confiável que
  o normal perto de 0 e de 1.
* Na calibração, no viés e no retorno por faixa, cada partida contribui com três
  observações **correlacionadas**. Por isso os erros padrão da regressão
  logística e do retorno médio são **agrupados por partida**; ignorar isso
  produziria intervalos artificialmente estreitos.
* O teste do viés favorito-azarão usa as probabilidades **normalizadas**, com a
  margem removida. É a escolha conservadora: boa parte do que costuma ser lido
  como viés favorito-azarão vem da própria margem, que incide mais pesadamente
  sobre as cotações altas.
* A comparação com o mercado usa **bootstrap pareado** e **McNemar**, ambos
  pareados nas mesmas partidas de teste.
* O **log loss é calculado por uma função própria** (`utils.log_loss_ordenado`)
  em vez de `sklearn.metrics.log_loss`. O motivo é uma armadilha real: o
  scikit-learn ordena o argumento `labels` **alfabeticamente** e pressupõe que
  as colunas de `y_pred` sigam essa ordem (A, D, H). Como todo o notebook
  trabalha na ordem fixa **H, D, A**, chamar a função do scikit-learn trocaria
  em silêncio as colunas de vitória do mandante e do visitante — o log loss
  sairia errado e, pior, a escolha de hiperparâmetros seria feita com base
  nele. A função própria recebe a ordem das classes explicitamente.

### Modelagem

* A divisão é **temporal** e nunca aleatória; o conjunto de teste é usado uma
  única vez, no fim.
* Imputação, padronização e hiperparâmetros são ajustados **apenas no treino**,
  dentro de `Pipeline`s do scikit-learn.
* As grades de hiperparâmetros são pequenas de propósito, para manter a execução
  do notebook abaixo de 15 minutos numa máquina comum. Grades maiores poderiam
  render ganhos marginais.
* O alvo tem **três classes desbalanceadas** e o empate é estruturalmente difícil
  de prever: ele raramente é o desfecho mais provável, então quase nunca é
  escolhido — nem pelos modelos, nem pelo mercado. Isso é uma característica do
  problema, não um defeito dos modelos.